# B5 · Modèles de diffusion : un DDPM minimal — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/b5_diffusion/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou des tests). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/b5_diffusion/03_notebook.ipynb`, créée par `python tools/start_chapter.py B5`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | B5.Q1 à B5.Q11, B5.R1 à B5.R3, B5.1 à B5.5 | vérifier tes réponses courtes | 🧠 🔁 ✏️ | ★ à ★★ | |
| A | B5.12 | linear_beta_schedule, cosine_beta_schedule et noise_schedule | 🔨 | ★★ | 20 |
|  | B5.13 | Prédire l'allure d'un chiffre bruité aux pas 100, 300 et 600, planning linéaire puis cosinus | 🔮 | ★ | 10 |
|  | B5.14 | q_sample : bruiter un batch d'images à des pas différents | 🔨 | ★★ | 20 |
|  | B5.15 | Lire ᾱ_t, le rapport signal sur bruit et la grille de bruitage des deux plannings | 📈 | ★ | 15 |
|  | B5.16 | q_posterior_mean_variance et predict_x0_from_eps | 🔨 | ★★ | 30 |
|  | B5.17 | ddpm_step et ddim_step | 🔨 | ★★★ | 40 |
|  | B5.18 | Avec le vrai bruit, DDPM et DDIM retrouvent-ils x_0 ? Contrôle sur des données 2D | 🔬 | ★★ | 25 |
| B | B5.19 | Diffusion en 2D : un MLP débruiteur conditionné par le pas de temps, entraîné puis échantillonné | 📦 | ★★ | 30 |
|  | B5.20 | Flow matching en 2D : même MLP, cible « vitesse » et intégration d'Euler | 🔬 | ★★★ | 40 |
| C | B5.21 | Un U-Net minuscule en PyTorch : blocs convolutifs, connexions de saut, encodage du pas de temps | 📦 | ★★★ | 50 |
| C | B5.22 à B5.31 | la suite du U-Net sur MNIST et Fashion-MNIST : quatre bugs, entraînement avec EMA, DDPM contre DDIM, nombre de pas, guidance, mémorisation, distance de Fréchet, script autonome et défi (*prochaine étape de la génération*) | | | |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="B5")  # your own library

## Objectifs et rappel express

- Dériver la forme fermée du bruitage et le posterior d'un pas, et calculer à la main un pas DDPM et un pas DDIM.
- Implémenter en NumPy les plannings de bruit, le bruitage et les pas d'échantillonnage, et les contrôler par l'aller-retour avec le vrai bruit.
- Écrire et entraîner un petit U-Net conditionné par le pas, l'échantillonner par DDPM et DDIM, et le conditionner par la classe avec la guidance sans classifieur.
- Évaluer un générateur (distance de Fréchet, plus proches voisins) et le situer face au VAE et au GAN.

**Rappel express.** Pas de l'article $t = 1..T$, indice du code $t - 1$. $\alpha_t = 1 - \beta_t$, $\bar\alpha_t = \prod_{s \le t} \alpha_s$, $\mathbf{x}_t = \sqrt{\bar\alpha_t}\,\mathbf{x}_0 + \sqrt{1 - \bar\alpha_t}\,\boldsymbol{\varepsilon}$, $\operatorname{SNR}(t) = \bar\alpha_t / (1 - \bar\alpha_t)$. Posterior : $\tilde\beta_t = \frac{1 - \bar\alpha_{t-1}}{1 - \bar\alpha_t}\beta_t$. DDPM : $\mathbf{x}_{t-1} = \frac{1}{\sqrt{\alpha_t}}\big(\mathbf{x}_t - \frac{\beta_t}{\sqrt{1 - \bar\alpha_t}}\boldsymbol{\varepsilon}_\theta\big) + \sigma_t\mathbf{z}$, sans bruit au dernier pas. DDIM ($\eta = 0$) : $\hat{\mathbf{x}}_0 = (\mathbf{x}_t - \sqrt{1 - \bar\alpha_t}\,\boldsymbol{\varepsilon}_\theta) / \sqrt{\bar\alpha_t}$, puis $\mathbf{x}_{t'} = \sqrt{\bar\alpha_{t'}}\,\hat{\mathbf{x}}_0 + \sqrt{1 - \bar\alpha_{t'}}\,\boldsymbol{\varepsilon}_\theta$. Guidance : $\tilde{\boldsymbol{\varepsilon}} = (1 + w)\,\boldsymbol{\varepsilon}_\theta(\mathbf{x}_t, c) - w\,\boldsymbol{\varepsilon}_\theta(\mathbf{x}_t, \varnothing)$, `guidance_scale` $= 1 + w$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ B5.1 à B5.5)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; un vecteur : une liste de nombres (`[0.5, -1.25]`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », les dérivations ∂ B5.6 et B5.7, l'estimation 🧮 B5.8, la réflexion (🗣️ B5.9, ⚖️ B5.10, 📄 B5.11) et l'entretien se corrigent avec `05_solutions.md`.

**B5.Q1 — Diffusion, VAE, GAN : comment chacun fabrique-t-il une image ?**

In [ ]:
# B5.Q1: the values you found on paper
answer_B5_Q1a = ...  # a) the letter of your choice, e.g. "E"
answer_B5_Q1b = ...  # b) True or False
answer_B5_Q1c = ...  # c) the letter of your choice, e.g. "E"
answer_B5_Q1d = ...  # d) True or False
answer_B5_Q1e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_B5_Q1a, answer_B5_Q1b, answer_B5_Q1c, answer_B5_Q1d, answer_B5_Q1e]):
    wb.check(f"B5.Q1{letter}", answer)

**B5.Q2 — Pourquoi détruire l'image en mille petits pas de bruit plutôt qu'en un seul ?**

In [ ]:
# B5.Q2: the values you found on paper
answer_B5_Q2a = ...  # a) the letter of your choice, e.g. "E"
answer_B5_Q2b = ...  # b) a number, 2 decimals
answer_B5_Q2c = ...  # c) a number, 2 decimals
answer_B5_Q2d = ...  # d) True or False
answer_B5_Q2e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B5_Q2a, answer_B5_Q2b, answer_B5_Q2c, answer_B5_Q2d, answer_B5_Q2e]):
    wb.check(f"B5.Q2{letter}", answer)

**B5.Q3 — ᾱ_t : quelle part de l'image reste-t-il au pas t ?**

In [ ]:
# B5.Q3: the values you found on paper
answer_B5_Q3a = ...  # a) a number, 2 decimals
answer_B5_Q3b = ...  # b) a number, 2 decimals
answer_B5_Q3c = ...  # c) True or False
answer_B5_Q3d = ...  # d) a number, 2 decimals
answer_B5_Q3e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B5_Q3a, answer_B5_Q3b, answer_B5_Q3c, answer_B5_Q3d, answer_B5_Q3e]):
    wb.check(f"B5.Q3{letter}", answer)

**B5.Q4 — Prédire le bruit, l'image ou la vitesse : la même information ?**

In [ ]:
# B5.Q4: the values you found on paper
answer_B5_Q4a = ...  # a) the letter of your choice, e.g. "E"
answer_B5_Q4b = ...  # b) a number, 3 decimals
answer_B5_Q4c = ...  # c) True or False
answer_B5_Q4d = ...  # d) the letter of your choice, e.g. "E"
answer_B5_Q4e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B5_Q4a, answer_B5_Q4b, answer_B5_Q4c, answer_B5_Q4d, answer_B5_Q4e]):
    wb.check(f"B5.Q4{letter}", answer)

**B5.Q5 — Le pas de temps est une entrée du réseau : pourquoi, et comment l'encoder ?**

In [ ]:
# B5.Q5: the values you found on paper
answer_B5_Q5a = ...  # a) the letter of your choice, e.g. "E"
answer_B5_Q5b = ...  # b) True or False
answer_B5_Q5c = ...  # c) a whole number
answer_B5_Q5d = ...  # d) the letter of your choice, e.g. "E"
answer_B5_Q5e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B5_Q5a, answer_B5_Q5b, answer_B5_Q5c, answer_B5_Q5d, answer_B5_Q5e]):
    wb.check(f"B5.Q5{letter}", answer)

**B5.Q6 — U-Net : à quoi servent les connexions de saut ?**

In [ ]:
# B5.Q6: the values you found on paper
answer_B5_Q6a = ...  # a) the letter of your choice, e.g. "E"
answer_B5_Q6b = ...  # b) a whole number
answer_B5_Q6c = ...  # c) True or False
answer_B5_Q6d = ...  # d) a whole number
answer_B5_Q6e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B5_Q6a, answer_B5_Q6b, answer_B5_Q6c, answer_B5_Q6d, answer_B5_Q6e]):
    wb.check(f"B5.Q6{letter}", answer)

**B5.Q7 — Échantillonnage DDPM : pourquoi est-il lent et aléatoire ?**

In [ ]:
# B5.Q7: the values you found on paper
answer_B5_Q7a = ...  # a) a whole number
answer_B5_Q7b = ...  # b) True or False
answer_B5_Q7c = ...  # c) True or False
answer_B5_Q7d = ...  # d) the letter of your choice, e.g. "E"
answer_B5_Q7e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B5_Q7a, answer_B5_Q7b, answer_B5_Q7c, answer_B5_Q7d, answer_B5_Q7e]):
    wb.check(f"B5.Q7{letter}", answer)

**B5.Q8 — DDIM : moins de pas avec le même réseau, vrai ou faux**

In [ ]:
# B5.Q8: the values you found on paper
answer_B5_Q8a = ...  # a) True or False
answer_B5_Q8b = ...  # b) True or False
answer_B5_Q8c = ...  # c) True or False
answer_B5_Q8d = ...  # d) True or False
answer_B5_Q8e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B5_Q8a, answer_B5_Q8b, answer_B5_Q8c, answer_B5_Q8d, answer_B5_Q8e]):
    wb.check(f"B5.Q8{letter}", answer)

**B5.Q9 — Guidance sans classifieur : ce que règle l'échelle w**

In [ ]:
# B5.Q9: the values you found on paper
answer_B5_Q9a = ...  # a) a number, 1 decimal
answer_B5_Q9b = ...  # b) the letter of your choice, e.g. "E"
answer_B5_Q9c = ...  # c) True or False
answer_B5_Q9d = ...  # d) a whole number
answer_B5_Q9e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B5_Q9a, answer_B5_Q9b, answer_B5_Q9c, answer_B5_Q9d, answer_B5_Q9e]):
    wb.check(f"B5.Q9{letter}", answer)

**B5.Q10 — Un bon FID peut-il cacher de la copie des données d'entraînement ?**

In [ ]:
# B5.Q10: the values you found on paper
answer_B5_Q10a = ...  # a) True or False
answer_B5_Q10b = ...  # b) the letter of your choice, e.g. "E"
answer_B5_Q10c = ...  # c) True or False
answer_B5_Q10d = ...  # d) the letter of your choice, e.g. "E"
answer_B5_Q10e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B5_Q10a, answer_B5_Q10b, answer_B5_Q10c, answer_B5_Q10d, answer_B5_Q10e]):
    wb.check(f"B5.Q10{letter}", answer)

**B5.Q11 — Diffusion latente, transformers de diffusion, flow matching : ce qui a changé depuis 2020**

In [ ]:
# B5.Q11: the values you found on paper
answer_B5_Q11a = ...  # a) a whole number
answer_B5_Q11b = ...  # b) the letter of your choice, e.g. "E"
answer_B5_Q11c = ...  # c) the letter of your choice, e.g. "E"
answer_B5_Q11d = ...  # d) True or False
answer_B5_Q11e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B5_Q11a, answer_B5_Q11b, answer_B5_Q11c, answer_B5_Q11d, answer_B5_Q11e]):
    wb.check(f"B5.Q11{letter}", answer)

**B5.R1 — L'encodage sinusoïdal des positions**

In [ ]:
# B5.R1: the values you found on paper
answer_B5_R1a = ...  # a) a whole number
answer_B5_R1b = ...  # b) a number, 2 decimals
answer_B5_R1c = ...  # c) True or False
answer_B5_R1d = ...  # d) True or False
answer_B5_R1e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B5_R1a, answer_B5_R1b, answer_B5_R1c, answer_B5_R1d, answer_B5_R1e]):
    wb.check(f"B5.R1{letter}", answer)

**B5.R2 — Ce que mesure la distance de Fréchet et l'effondrement de mode**

In [ ]:
# B5.R2: the values you found on paper
answer_B5_R2a = ...  # a) a whole number
answer_B5_R2b = ...  # b) the letter of your choice, e.g. "E"
answer_B5_R2c = ...  # c) True or False
answer_B5_R2d = ...  # d) True or False
answer_B5_R2e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_B5_R2a, answer_B5_R2b, answer_B5_R2c, answer_B5_R2d, answer_B5_R2e]):
    wb.check(f"B5.R2{letter}", answer)

**B5.R3 — L'astuce de reparamétrisation et le terme KL du VAE**

In [ ]:
# B5.R3: the values you found on paper
answer_B5_R3a = ...  # a) a number, 2 decimals
answer_B5_R3b = ...  # b) a number, 4 decimals
answer_B5_R3c = ...  # c) the letter of your choice, e.g. "E"
answer_B5_R3d = ...  # d) True or False
answer_B5_R3e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B5_R3a, answer_B5_R3b, answer_B5_R3c, answer_B5_R3d, answer_B5_R3e]):
    wb.check(f"B5.R3{letter}", answer)

**Ex B5.1 — Guidance sans classifieur : combiner les deux prédictions pour w = 0, 1 et 3**

In [ ]:
# Ex B5.1: the values you found on paper
answer_B5_1a = ...  # a) a list of two numbers, 2 decimals
answer_B5_1b = ...  # b) a list of two numbers, 2 decimals
answer_B5_1c = ...  # c) a list of two numbers, 2 decimals
answer_B5_1d = ...  # d) a whole number
answer_B5_1e = ...  # e) a number, 2 decimals
answer_B5_1f = ...  # f) True or False
answer_B5_1g = ...  # g) True or False
answer_B5_1h = ...  # h) a whole number

for letter, answer in zip("abcdefgh", [answer_B5_1a, answer_B5_1b, answer_B5_1c, answer_B5_1d, answer_B5_1e, answer_B5_1f, answer_B5_1g, answer_B5_1h]):
    wb.check(f"B5.1{letter}", answer)

**Ex B5.2 — Planning linéaire : β_t, α_t et ᾱ_t pour T = 4, puis ᾱ_T pour T = 1 000**

In [ ]:
# Ex B5.2: the values you found on paper
answer_B5_2a = ...  # a) a number, 2 decimals
answer_B5_2b = ...  # b) a number, 4 decimals
answer_B5_2c = ...  # c) a number, 4 decimals
answer_B5_2d = ...  # d) a number, 4 decimals
answer_B5_2e = ...  # e) a number, 4 decimals
answer_B5_2f = ...  # f) True or False
answer_B5_2g = ...  # g) a number, 2 decimals
answer_B5_2h = ...  # h) a number, 2 decimals
answer_B5_2i = ...  # i) a number, 2 decimals
answer_B5_2j = ...  # j) a number, 2 decimals
answer_B5_2k = ...  # k) a number, 2 decimals
answer_B5_2l = ...  # l) True or False

for letter, answer in zip("abcdefghijkl", [answer_B5_2a, answer_B5_2b, answer_B5_2c, answer_B5_2d, answer_B5_2e, answer_B5_2f, answer_B5_2g, answer_B5_2h, answer_B5_2i, answer_B5_2j, answer_B5_2k, answer_B5_2l]):
    wb.check(f"B5.2{letter}", answer)

**Ex B5.3 — Rapport signal sur bruit : à quel pas un chiffre devient-il illisible, planning linéaire ou cosinus ?**

In [ ]:
# Ex B5.3: the values you found on paper
answer_B5_3a = ...  # a) a number, 3 decimals
answer_B5_3b = ...  # b) a number, 2 decimals
answer_B5_3c = ...  # c) a number, 2 decimals
answer_B5_3d = ...  # d) a number, 2 decimals
answer_B5_3e = ...  # e) a number, 2 decimals
answer_B5_3f = ...  # f) a number, 2 decimals
answer_B5_3g = ...  # g) a number, 2 decimals
answer_B5_3h = ...  # h) True or False
answer_B5_3i = ...  # i) True or False
answer_B5_3j = ...  # j) the letter of your choice, e.g. "E"

for letter, answer in zip("abcdefghij", [answer_B5_3a, answer_B5_3b, answer_B5_3c, answer_B5_3d, answer_B5_3e, answer_B5_3f, answer_B5_3g, answer_B5_3h, answer_B5_3i, answer_B5_3j]):
    wb.check(f"B5.3{letter}", answer)

**Ex B5.4 — Un pas DDPM à la main, en dimension 1, avec un bruit prédit donné**

In [ ]:
# Ex B5.4: the values you found on paper
answer_B5_4a = ...  # a) a number, 4 decimals
answer_B5_4b = ...  # b) a number, 4 decimals
answer_B5_4c = ...  # c) a number, 4 decimals
answer_B5_4d = ...  # d) a number, 4 decimals
answer_B5_4e = ...  # e) a number, 4 decimals
answer_B5_4f = ...  # f) a number, 4 decimals
answer_B5_4g = ...  # g) a number, 4 decimals
answer_B5_4h = ...  # h) True or False
answer_B5_4i = ...  # i) True or False

for letter, answer in zip("abcdefghi", [answer_B5_4a, answer_B5_4b, answer_B5_4c, answer_B5_4d, answer_B5_4e, answer_B5_4f, answer_B5_4g, answer_B5_4h, answer_B5_4i]):
    wb.check(f"B5.4{letter}", answer)

**Ex B5.5 — Un pas DDIM à la main et l'image estimée x̂_0**

In [ ]:
# Ex B5.5: the values you found on paper
answer_B5_5a = ...  # a) a number, 4 decimals
answer_B5_5b = ...  # b) a number, 4 decimals
answer_B5_5c = ...  # c) the letter of your choice, e.g. "E"
answer_B5_5d = ...  # d) a number, 4 decimals
answer_B5_5e = ...  # e) True or False
answer_B5_5f = ...  # f) a number, 4 decimals
answer_B5_5g = ...  # g) a number, 4 decimals
answer_B5_5h = ...  # h) a number, 4 decimals
answer_B5_5i = ...  # i) True or False

for letter, answer in zip("abcdefghi", [answer_B5_5a, answer_B5_5b, answer_B5_5c, answer_B5_5d, answer_B5_5e, answer_B5_5f, answer_B5_5g, answer_B5_5h, answer_B5_5i]):
    wb.check(f"B5.5{letter}", answer)

## Partie A · Les mathématiques de ta librairie et leurs contrôles

Tu écris dans `mylearn/diffusion.py` tout ce qui, dans un modèle de diffusion, ne dépend pas du réseau : les plannings de bruit et leurs constantes (B5.12), le bruitage en forme fermée (B5.14), le posterior et l'estimation de $\mathbf{x}_0$ (B5.16), puis un pas DDPM et un pas DDIM (B5.17). Entre deux, tu prédis ce qu'un classifieur voit encore d'un chiffre bruité (B5.13) et tu lis les plannings sur une figure (B5.15). Pour finir, tu enchaînes tes pas sur des points 2D avec le vrai bruit, le contrôle qui attrape les erreurs d'indice et de racine avant tout entraînement (B5.18).

**Contrat commun de `diffusion.py`** (les docstrings, plus ces précisions que vérifient les tests) :
- les pas sont des **indices** de 0 à $T - 1$ : le pas $t$ de l'article, celui des formules de la fiche, est l'indice $t - 1$ ; un indice $-1$ ou $T$ lève `ValueError`, jamais `IndexError` (−1 n'est pas « le dernier pas ») ;
- tes fonctions ne modifient jamais leurs entrées (pas de `-=` ni de `*=` sur un tableau reçu) ; les entrées peuvent être des listes ;
- une valeur hors de son intervalle lève `ValueError`, **`nan` compris** : `nan` ne vérifie aucune condition « 0 < β < 1 », il doit donc être refusé ;
- tolérances des tests : $10^{-9}$ en relatif pour les plannings (des float64, par contrat), $10^{-6}$ en relatif et en absolu ailleurs (un calcul en float32 passe) ;
- si la bibliothèque diffusers est installée (elle l'est sur Colab), 25 tests comparent aussi tes fonctions à ses `DDPMScheduler` et `DDIMScheduler` ; sans elle, ils sont ignorés et les formes fermées restent testées.

La cellule suivante charge les outils des parties A et B : `TEST_DIFF` (le fichier des tests), `T_STEPS` (le nombre de pas du notebook : 400 en mode rapide, 1 000 en mode complet), `BETAS` (le planning linéaire du notebook, aux bornes multipliées par $1\,000/T$ : avec les bornes par défaut et $T = 400$, il resterait trop d'image au dernier pas, le piège « SNR final non nul » de la fiche), `DIGITS` (1 000 chiffres de test de MNIST, ramenés dans $[-1, 1]$ par `wbd.to_signed`, toujours les mêmes), `toy_schedule` (les neuf constantes de `noise_schedule`, écrites sans ton code, pour les vérifications) et `show_images`. Le module `wbd` (`wb.diffusion`) contient l'infrastructure du chapitre : données, plannings, jouets 2D et modèles de référence ; le notebook ne s'en sert que pour vérifier ton code ou quand il manque.

In [ ]:
# Tools of parts A and B: your NumPy library diffusion.py, then the 2-D toys
import copy
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from wb import diffusion as wbd
from wb.nbtools import (StopCheck, checked, filled, fr, pair, refuses, returned, run_pytest_group, same, shape_text,
                        verdict, written)

TEST_DIFF = "tests/test_chb5_diffusion.py"          # the tests of your library diffusion.py
T_STEPS = wbd.n_steps(FAST_MODE)                     # 400 steps in fast mode, 1 000 in full mode
BETAS = wbd.schedule_betas("linear", T_STEPS)        # linear, bounds x 1000 / T (computed here, without your code)
DIGITS = wbd.image_subset("mnist", 1000, split="test")   # 1 000 MNIST test digits in [-1, 1], always the same


def fr_e(x):
    """A small number in French scientific notation: 3,6e-05."""
    return f"{x:.1e}".replace(".", ",")


def toy_schedule(betas):
    """The nine constants of noise_schedule (B5.12), written once without your code, for the checks."""
    b = np.asarray(betas, dtype=float)
    a = 1 - b
    ab = np.cumprod(a)
    ab_prev = np.concatenate([[1.0], ab[:-1]])
    return {"betas": b, "alphas": a, "alpha_bars": ab, "alpha_bars_prev": ab_prev, "sqrt_alpha_bars": np.sqrt(ab),
            "sqrt_one_minus_alpha_bars": np.sqrt(1 - ab), "posterior_variance": b * (1 - ab_prev) / (1 - ab),
            "posterior_mean_coef1": b * np.sqrt(ab_prev) / (1 - ab),
            "posterior_mean_coef2": (1 - ab_prev) * np.sqrt(a) / (1 - ab)}


def show_images(images, titles=None, ncol=None, size=1.3, suptitle=None):
    """Images in [-1, 1] (a tensor or an array (N, 1, 28, 28)) in a grid of ncol columns."""
    arr = images.detach().cpu().numpy() if torch.is_tensor(images) else np.asarray(images, dtype=float)
    arr = wbd.to_unit(arr.reshape(-1, 28, 28))
    ncol = ncol or len(arr)
    nrow = math.ceil(len(arr) / ncol)
    fig, axes = plt.subplots(nrow, ncol, figsize=(size * ncol, size * nrow + (0.4 if suptitle else 0.1)), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for i, img in enumerate(arr):
        axes.ravel()[i].imshow(img, cmap="gray", vmin=0, vmax=1)
        if titles is not None:
            axes.ravel()[i].set_title(titles[i], fontsize=8)
    if suptitle:
        fig.suptitle(suptitle)
    plt.tight_layout()
    plt.show()


print(f"T = {T_STEPS} pas en {'mode rapide' if FAST_MODE else 'mode complet'} ; planning linéaire de bornes "
      f"{fr_e(BETAS[0])} et {fr_e(BETAS[-1])} : ᾱ au dernier indice = {fr_e(wbd.alpha_bars(BETAS)[-1])}. "
      f"{len(DIGITS)} chiffres de test de MNIST dans [−1, 1].")

### Ex B5.12 — linear_beta_schedule, cosine_beta_schedule et noise_schedule 🔨 ★★ ⏱️ 20 min
**Objectif :** écrire les deux plannings de bruit et toutes les constantes qui en dérivent.  
**Prérequis :** ✏️ B5.2 · fiche §205.2 (« Le planning linéaire de Ho et coll. », « Le planning cosinus de Nichol et Dhariwal », « Bruiter un batch : `noise_schedule` et `q_sample` ») · **mylearn :** `diffusion.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** : ouvre `mon_travail/mylearn/diffusion.py` (créé par `python tools/start_chapter.py B5`), lis la docstring, remplace chaque `raise NotImplementedError(...)` de l'exercice par ton code et **enregistre**. NumPy seulement. La première cellule de vérification recharge ta librairie et l'essaie sur des cas visibles ; la seconde lance les tests de l'exercice. `python -m pytest tests/test_chb5_diffusion.py -q` les lance tous dans un terminal.

**Contrat**
- `linear_beta_schedule(n_steps, beta_start=1e-4, beta_end=0.02)` : un tableau float64 de forme `(T,)`, de `beta_start` (indice 0) à `beta_end` (indice $T - 1$), régulièrement espacés ; pour `n_steps = 1`, `[beta_start]`, comme `np.linspace` (aucune division par $T - 1$) ; `ValueError` si `n_steps < 1` ou si $0 < \text{beta\_start} \le \text{beta\_end} < 1$ n'est pas vérifié (`nan` compris).
- `cosine_beta_schedule(n_steps, s=0.008, max_beta=0.999)` : la fonction $f$ de la docstring évaluée aux $T + 1$ points $t/T$, $t = 0 \dots T$ (ni $T$ points, ni des points de $[0, 1)$), puis $\beta_t = \min(1 - f(t+1)/f(t),\ \text{max\_beta})$ pour $t = 0 \dots T - 1$, en float64 ; l'écrêtage à `max_beta` est atteint au dernier indice ; `ValueError` si `n_steps < 1`, `s < 0` ou `max_beta` hors de $(0, 1)$ (`nan` compris).
- `noise_schedule(betas)` : un dictionnaire aux neuf clés de la docstring, chacune un tableau float64 de forme `(T,)`, **même pour des betas en float32** ; `alpha_bars_prev[0]` vaut 1, donc `posterior_variance[0]` vaut 0 **exactement** ; `ValueError` si `betas` n'est pas 1-D ou si un $\beta$ est hors de $(0, 1)$, `nan` compris.

La première cellule de vérification essaie tes trois fonctions sur les exemples des docstrings, contre `np.linspace` et la forme fermée du cosinus, sur le cas $T = 1$, l'écrêtage et six refus, puis affiche $\bar\alpha$ au dernier indice du planning linéaire de $T$ = `T_STEPS` pas, avec les bornes par défaut puis multipliées par $1\,000/T$ (rien à taper). La seconde lance les 60 tests de l'exercice (`-k "test_linear_beta_schedule_ or test_cosine_beta_schedule_ or test_noise_schedule_"`).

Dans tes notes : pourquoi `posterior_variance[0]` vaut-il 0, et qu'en conclure pour le dernier pas de l'échantillonnage ? Avec $T = 400$, que vaut $\bar\alpha$ au dernier indice avec les bornes par défaut, quelle part de l'image reste-t-il (son coefficient $\sqrt{\bar\alpha}$), et pourquoi multiplier les bornes par $1\,000/T$ règle-t-il le problème (✏️ B5.2) ? Pourquoi un test « aucun $\beta$ n'est $\le 0$ ni $\ge 1$ » laisse-t-il passer un `nan` ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="B5")   # reload your saved file
with checked("B5.12"):
    M = mylearn.diffusion
    lin_12 = M.linear_beta_schedule(5)
    if returned("B5.12", "linear_beta_schedule", lin_12):
        verdict("B5.12", np.asarray(lin_12).dtype == np.float64, "linear_beta_schedule rend des float64.",
                f"linear_beta_schedule doit rendre des float64, reçu {np.asarray(lin_12).dtype}.")
        same("B5.12", "linear_beta_schedule(5) (exemple de la docstring)", lin_12, np.linspace(1e-4, 0.02, 5))
    same("B5.12", f"linear_beta_schedule({T_STEPS}, …) aux bornes × 1000/T, contre np.linspace",
         M.linear_beta_schedule(T_STEPS, 1e-4 * 1000 / T_STEPS, 0.02 * 1000 / T_STEPS), BETAS)
    same("B5.12", "linear_beta_schedule(1, 0.3, 0.5) : un seul pas, beta_start comme np.linspace",
         M.linear_beta_schedule(1, 0.3, 0.5), [0.3])
    refuses("B5.12", "linear_beta_schedule(10, beta_start=nan)", lambda: M.linear_beta_schedule(10, float("nan")))
    refuses("B5.12", "linear_beta_schedule(10, 0.02, 0.01) (beta_start > beta_end)",
            lambda: M.linear_beta_schedule(10, 0.02, 0.01))
    refuses("B5.12", "linear_beta_schedule(0)", lambda: M.linear_beta_schedule(0))

    f_12 = np.cos((np.arange(6) / 5 + 0.008) / 1.008 * np.pi / 2) ** 2          # f at the T + 1 = 6 points t / T
    same("B5.12", "cosine_beta_schedule(5) contre la forme fermée", M.cosine_beta_schedule(5),
         np.minimum(1 - f_12[1:] / f_12[:-1], 0.999))
    same("B5.12", "cosine_beta_schedule(1000) contre la forme fermée", M.cosine_beta_schedule(1000),
         wbd.schedule_betas("cosine", 1000))
    same("B5.12", "cosine_beta_schedule(8, max_beta=0.3) : l'écrêtage atteint au dernier indice",
         np.asarray(M.cosine_beta_schedule(8, max_beta=0.3))[-1], 0.3)
    refuses("B5.12", "cosine_beta_schedule(10, max_beta=1.0)", lambda: M.cosine_beta_schedule(10, max_beta=1.0))
    refuses("B5.12", "cosine_beta_schedule(10, s=-0.1)", lambda: M.cosine_beta_schedule(10, s=-0.1))

    betas_12 = np.array([0.1, 0.2, 0.5])
    sched_12 = M.noise_schedule(betas_12)
    if returned("B5.12", "noise_schedule", sched_12):
        want_12 = toy_schedule(betas_12)
        keys_12 = set(sched_12) if isinstance(sched_12, dict) else set()
        verdict("B5.12", keys_12 == set(want_12), "noise_schedule rend les neuf clés de la docstring.",
                f"noise_schedule doit rendre un dictionnaire aux clés {sorted(want_12)} ; manquent "
                f"{sorted(set(want_12) - keys_12)}, en trop {sorted(keys_12 - set(want_12))}.")
        for name_12 in sorted(set(want_12) & keys_12):
            same("B5.12", f"noise_schedule([0.1, 0.2, 0.5])[{name_12!r}]", sched_12[name_12], want_12[name_12])
        if "posterior_variance" in keys_12:
            verdict("B5.12", np.asarray(sched_12["posterior_variance"])[0] == 0,
                    "posterior_variance[0] vaut 0 exactement.",
                    "posterior_variance[0] doit valoir 0 exactement (alpha_bars_prev[0] = 1).")
    sched32_12 = M.noise_schedule(np.array([0.1, 0.2], dtype=np.float32))
    verdict("B5.12", all(np.asarray(v).dtype == np.float64 for v in sched32_12.values()),
            "des betas en float32 donnent des constantes en float64.",
            "noise_schedule doit rendre des float64, même pour des betas en float32.")
    verdict("B5.12", np.array_equal(betas_12, [0.1, 0.2, 0.5]), "les betas reçus n'ont pas changé.",
            "noise_schedule a modifié les betas reçus.")
    refuses("B5.12", "noise_schedule([0.1, nan])", lambda: M.noise_schedule([0.1, float("nan")]))
    refuses("B5.12", "noise_schedule([0.1, 1.0])", lambda: M.noise_schedule([0.1, 1.0]))
    refuses("B5.12", "noise_schedule([[0.1, 0.2]]) (2-D)", lambda: M.noise_schedule([[0.1, 0.2]]))

    end_default_12 = float(M.noise_schedule(M.linear_beta_schedule(T_STEPS))["alpha_bars"][-1])
    end_scaled_12 = float(M.noise_schedule(BETAS)["alpha_bars"][-1])
    print(f"ᾱ au dernier indice, T = {T_STEPS} : bornes par défaut {fr_e(end_default_12)}, bornes × 1000/T "
          f"{fr_e(end_scaled_12)} (avec ta librairie).")

In [ ]:
if written(lambda: mylearn.diffusion.linear_beta_schedule(2)):
    run_pytest_group(TEST_DIFF, "test_linear_beta_schedule_ or test_cosine_beta_schedule_ or test_noise_schedule_")   # 60 tests
else:
    print("⏳ Ex B5.12 : les tests attendent tes fonctions.")

### Ex B5.13 — Prédire l'allure d'un chiffre bruité aux pas 100, 300 et 600, planning linéaire puis cosinus 🔮 ★ ⏱️ 10 min
**Objectif :** prédire jusqu'à quel niveau de bruit un classifieur entraîné sur des chiffres propres reconnaît encore un chiffre bruité, et comparer à la convention de lisibilité de ✏️ B5.3.  
**Prérequis :** Ex B5.12 · ✏️ B5.3 · fiche §205.2 (« Le rapport signal sur bruit ») · ch. 27 (le classifieur de la partie B) · **Fil rouge :** MNIST · **Parcours :** R, C

En ✏️ B5.3, tu as adopté une **convention** : un chiffre reste lisible tant que son rapport signal sur bruit vaut au moins 0,1. Qu'en pense un vrai lecteur ? La cellule suivante prépare `CLF`, le petit CNN de la partie B du ch. 27 (`wbd.classifier`), entraîné une fois sur 10 000 chiffres **propres** de MNIST ; il en reconnaît environ 97 % sur les 1 000 chiffres de test. Elle donne aussi `noisy_13(planning, pas)`, qui bruite ces 1 000 chiffres par la forme fermée, au **pas de l'article** 100, 300 ou 600 (les indices 99, 299 et 599), du planning linéaire ou cosinus de $T = 1\,000$, toujours avec le même bruit.

L'expérience montre quatre chiffres à ces trois pas pour chaque planning, puis, pour chacun des six cas, le SNR et l'**accuracy** du classifieur sur les chiffres bruités. Tu peux calculer les six SNR (✏️ B5.3), pas l'accuracy. **Avant d'exécuter**, écris tes deux prédictions :
- a) `strong_13` : dans combien des six cas (3 pas × 2 plannings) le classifieur reconnaît-il **au moins 70 %** des chiffres ? Un entier de 0 à 6 ;
- b) `cos600_13` : au pas 600 du planning cosinus, où le SNR dépasse largement le seuil 0,1 de ta convention, le classifieur reconnaît-il **plus de la moitié** des chiffres ? `True` ou `False`.

La cellule d'expérience n'affiche rien tant que tes deux prédictions ne sont pas écrites ; elle les vérifie à la fin. Rien n'y dépend de ton code.

In [ ]:
CLF = wbd.classifier("mnist")                      # the CNN of ch. 27, trained once on clean digits, then reloaded
STEPS_13 = (100, 300, 600)                          # steps of the paper: indices 99, 299, 599
AB_13 = {"linéaire": wbd.alpha_bars(wbd.schedule_betas("linear", 1000)),
         "cosinus": wbd.alpha_bars(wbd.schedule_betas("cosine", 1000))}
EPS_13 = np.random.default_rng(13).standard_normal(tuple(DIGITS.images.shape))   # one noise per digit, for every case
X0_13 = DIGITS.images.double().numpy()


def noisy_13(kind, step):
    """The 1 000 test digits at the step `step` of the paper (index step - 1) of the schedule `kind` (T = 1 000),
    all with the same noise EPS_13 (the closed form, written here without your q_sample)."""
    a = AB_13[kind][step - 1]
    return math.sqrt(a) * X0_13 + math.sqrt(1 - a) * EPS_13


print(f"Le classifieur reconnaît {fr(100 * np.mean(wbd.predict(CLF, X0_13) == DIGITS.labels.numpy()), 1)} % des "
      "1 000 chiffres propres.")

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
strong_13 = ...    # a) in how many of the 6 cases (steps 100, 300, 600 x linear, cosine) does the classifier
                   #    recognise at least 70 % of the noisy digits? an integer from 0 to 6
cos600_13 = ...    # b) cosine schedule, step 600: does the classifier recognise MORE than half of the digits?
                   #    True or False

In [ ]:
if any(answer is ... or answer is None for answer in [strong_13, cos600_13]):
    print("⏳ Ex B5.13 : écris d'abord tes deux prédictions dans la cellule précédente.")
else:
    # 1. four digits, clean then at the steps 100, 300 and 600 of each schedule (the same noise everywhere)
    for kind_13 in AB_13:
        row_13 = [X0_13[:4]] + [noisy_13(kind_13, s)[:4] for s in STEPS_13]
        show_images(np.stack(row_13, axis=1).reshape(-1, 1, 28, 28), ncol=16,
                    titles=[f"{s}" for _ in range(4) for s in ("propre",) + STEPS_13], size=0.9,
                    suptitle=f"planning {kind_13}, T = 1 000 : chaque chiffre propre, puis aux pas 100, 300, 600")

    # 2. the accuracy of the classifier, and the SNR, in each of the 6 cases
    acc_13 = {}
    for kind_13 in AB_13:
        for step_13 in STEPS_13:
            a_13 = AB_13[kind_13][step_13 - 1]
            pred_13 = wbd.predict(CLF, noisy_13(kind_13, step_13))
            acc_13[kind_13, step_13] = float(np.mean(pred_13 == DIGITS.labels.numpy()))
            top_13 = np.bincount(pred_13, minlength=10)
            print(f"{kind_13:>9} pas {step_13} : SNR {fr(a_13 / (1 - a_13), 3):>7} · accuracy {fr(100 * acc_13[kind_13, step_13], 1):>5} % "
                  f"· classe la plus prédite : {top_13.argmax()} ({fr(100 * top_13.max() / len(pred_13), 0)} % des chiffres)")
    wb.check("B5.13a", strong_13)
    wb.check("B5.13b", cos600_13)

Dans tes notes : pour chaque prédiction ratée, qu'est-ce qui t'a trompé ? À partir de quel SNR, à peu près, le classifieur décroche-t-il, et ton œil, sur les images ? Quelle classe le classifieur prédit-il quand il ne sait plus, et pourquoi un réseau entraîné sur des images propres se trompe-t-il ainsi sur du bruit qu'il n'a jamais vu ? Qu'est-ce que cela dit du débruiteur d'un modèle de diffusion, qui, lui, voit des images de tous les niveaux de bruit pendant son entraînement ?

### Ex B5.14 — q_sample : bruiter un batch d'images à des pas différents 🔨 ★★ ⏱️ 20 min
**Objectif :** écrire le bruitage en forme fermée d'un batch où chaque exemple a son propre pas.  
**Prérequis :** Ex B5.12 · ∂ B5.6 · fiche §205.2 (« La forme fermée », « Bruiter un batch » et le piège « `alpha_bars[t]` sans remise en forme ») · **Fil rouge :** MNIST · **mylearn :** `diffusion.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** : ouvre `mon_travail/mylearn/diffusion.py` (créé par `python tools/start_chapter.py B5`), lis la docstring, remplace chaque `raise NotImplementedError(...)` de l'exercice par ton code et **enregistre**. NumPy seulement. La première cellule de vérification recharge ta librairie et l'essaie sur des cas visibles ; la seconde lance les tests de l'exercice. `python -m pytest tests/test_chb5_diffusion.py -q` les lance tous dans un terminal.

**Contrat** de `q_sample(x0, t, alpha_bars, noise=None, rng=None)`, qui rend le couple `(x_t, noise)` :
- $\mathbf{x}_t = \sqrt{\bar\alpha_{t}}\,\mathbf{x}_0 + \sqrt{1 - \bar\alpha_{t}}\,\boldsymbol{\varepsilon}$ pour chaque exemple, avec **son** indice `t[n]` ; le coefficient est remis en forme `(N, 1, …, 1)`, quel que soit le nombre d'axes de `x0` (1-D, 2-D, `(N, C, H, W)`) ;
- `t` a **exactement un indice entier par exemple**, de forme `(N,)` : un `t` de longueur 1 pour un batch de plusieurs exemples lève `ValueError`, **même si NumPy saurait le diffuser** ; un indice hors de $[0, T)$ aussi ;
- `noise`, s'il est donné, a **exactement** la forme de `x0` (un bruit de forme `(2,)` pour `x0` de forme `(3, 2)` est refusé, même s'il se diffuse) ; sinon, **un seul** appel `rng.standard_normal(x0.shape)`, et c'est ce bruit qui est rendu (la cible de l'entraînement) ; `rng=None` : `np.random.default_rng()`, sans graine.

La première cellule de vérification essaie ta fonction sur l'exemple de la docstring, sur 6 chiffres de MNIST `(6, 1, 28, 28)` à 6 indices différents, sur un `x0` en 1-D, sur le bruit tiré par `rng`, et quatre refus ; elle montre les 6 chiffres bruités. La seconde lance les 19 tests de l'exercice (`-k "test_q_sample_"`).

Dans tes notes : que se passe-t-il, sans erreur, si tu multiplies `x0` de forme `(3, 1, 4, 3)` par `np.sqrt(alpha_bars[t])` de forme `(3,)` sans remise en forme (fiche, piège « `alpha_bars[t]` sans remise en forme ») ? Pourquoi rendre le bruit tiré, et pas seulement $\mathbf{x}_t$ ? Pourquoi refuser un `t` de longueur 1 que NumPy diffuserait volontiers ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="B5")   # reload your saved file
xt_14 = None
with checked("B5.14"):
    M = mylearn.diffusion
    ab_14 = np.array([0.9, 0.72, 0.36])
    x0_14 = np.array([[1.0, -1.0], [0.5, 0.0]])
    out_14, noise_14 = pair("B5.14", "q_sample", M.q_sample(x0_14.copy(), [0, 2], ab_14, noise=np.ones((2, 2))),
                            names=("x_t", "noise"))
    same("B5.14", "q_sample de l'exemple de la docstring", out_14,
         np.sqrt(ab_14[[0, 2]])[:, None] * x0_14 + np.sqrt(1 - ab_14[[0, 2]])[:, None], rtol=1e-6, atol=1e-6)

    ab_nb = wbd.alpha_bars(BETAS)
    imgs_14 = DIGITS.images[:6].double().numpy()
    t_14 = np.array([0, 1, 50, 150, 300, T_STEPS - 1])
    eps_14 = np.random.default_rng(14).standard_normal(imgs_14.shape)
    copies_14 = (imgs_14.copy(), t_14.copy(), eps_14.copy(), ab_nb.copy())
    xt_14, n_14 = pair("B5.14", "q_sample", M.q_sample(imgs_14, t_14, ab_nb, noise=eps_14), names=("x_t", "noise"))
    want_14 = np.sqrt(ab_nb[t_14])[:, None, None, None] * imgs_14 + np.sqrt(1 - ab_nb[t_14])[:, None, None, None] * eps_14
    if not same("B5.14", "6 chiffres (6, 1, 28, 28), un pas par image", xt_14, want_14, rtol=1e-6, atol=1e-6):
        xt_14 = None
    same("B5.14", "le bruit rendu est celui reçu", n_14, eps_14, rtol=1e-6, atol=1e-6)
    verdict("B5.14", all(np.array_equal(a, b) for a, b in zip((imgs_14, t_14, eps_14, ab_nb), copies_14)),
            "les entrées n'ont pas changé.", "q_sample a modifié une de ses entrées (x0, t, noise ou alpha_bars).")
    same("B5.14", "x0 en 1-D (3 exemples d'une valeur)", M.q_sample(np.array([0.5, -0.5, 1.0]), [0, 1, 2], ab_14,
                                                                    noise=np.array([1.0, 2.0, 3.0]))[0],
         np.sqrt(ab_14) * [0.5, -0.5, 1.0] + np.sqrt(1 - ab_14) * [1.0, 2.0, 3.0], rtol=1e-6, atol=1e-6)
    x_r, n_r = pair("B5.14", "q_sample", M.q_sample(x0_14, [1, 1], ab_14, rng=np.random.default_rng(5)),
                    names=("x_t", "noise"))
    same("B5.14", "sans noise : le bruit de rng.standard_normal(x0.shape), rendu", n_r,
         np.random.default_rng(5).standard_normal((2, 2)), rtol=1e-6, atol=1e-6)
    refuses("B5.14", "t de longueur 1 pour 2 exemples", lambda: M.q_sample(x0_14, [1], ab_14, noise=np.ones((2, 2))))
    refuses("B5.14", "un bruit de forme (2,) pour x0 de forme (2, 2)",
            lambda: M.q_sample(x0_14, [0, 1], ab_14, noise=np.ones(2)))
    refuses("B5.14", "t = -1", lambda: M.q_sample(x0_14, [-1, 0], ab_14, noise=np.ones((2, 2))))
    refuses("B5.14", "t = T", lambda: M.q_sample(x0_14, [0, 3], ab_14, noise=np.ones((2, 2))))
if xt_14 is not None:
    show_images(xt_14, titles=[f"indice {t}" for t in t_14], size=1.5,
                suptitle=f"ta q_sample : 6 chiffres, un pas chacun (T = {T_STEPS})")

In [ ]:
if written(lambda: mylearn.diffusion.q_sample([0.0], [0], [0.5], noise=[0.0])):
    run_pytest_group(TEST_DIFF, "test_q_sample_")   # 19 tests
else:
    print("⏳ Ex B5.14 : les tests attendent ta fonction q_sample.")

### Ex B5.15 — Lire ᾱ_t, le rapport signal sur bruit et la grille de bruitage des deux plannings 📈 ★ ⏱️ 15 min
**Objectif :** lire sur une figure ce qui distingue les plannings, et voir un même chiffre se dissoudre dans le bruit.  
**Prérequis :** Ex B5.14 · ✏️ B5.3 · fiche §205.2 (« Le planning linéaire… », « Le planning cosinus… », « Le rapport signal sur bruit » et le piège « un SNR final qui n'est pas nul ») · **Fil rouge :** MNIST · **Parcours :** C

La figure trace $\bar\alpha$ (à gauche) et $\ln \operatorname{SNR}$ (à droite) de quatre plannings, en fonction de la **fraction de la chaîne** parcourue, (indice + 1)/$T$, pour les comparer malgré leurs $T$ différents : **L**, le linéaire de Ho et coll. ($T = 1\,000$) ; **C**, le cosinus ($T = 1\,000$) ; **D**, le linéaire raccourci à $T = 400$ avec les bornes par défaut ; **S**, le linéaire à $T = 400$ avec les bornes multipliées par $1\,000/T$, celui du notebook en mode rapide. Dessous, le même chiffre et le même bruit aux indices 0, 99, 199, …, 999 de L, puis de C. Rien ne dépend de ton code. Réponds en lisant la figure :
- a) `same_curve_15` : laquelle des deux courbes à $T = 400$ suit celle de L : `"D"` ou `"S"` ?
- b) `largest_end_15` : quel planning garde **le plus** de l'image à son dernier indice : `"L"`, `"C"`, `"D"` ou `"S"` ?
- c) `cos_above_15` : sur quelle part de la chaîne le $\ln \operatorname{SNR}$ de C est-il au-dessus de celui de L ? `"A"` moins de la moitié ; `"B"` entre la moitié et 90 % ; `"C"` plus de 95 %.

Dans tes notes : avec L, à partir de quelle colonne de la grille ne distingues-tu plus le chiffre, et avec C ? Que gagne le planning cosinus, en nombre de pas « utiles » (où il reste quelque chose à débruiter), d'après la figure et ✏️ B5.3 ? Pourquoi D est-il un piège, et que verrait un modèle entraîné avec lui, puis échantillonné à partir d'un bruit pur ?

In [ ]:
# The four schedules of the figure, against the fraction t / T of the chain (computed without your code)
SCHEDULES_15 = {"L": ("linéaire, T = 1 000", wbd.schedule_betas("linear", 1000)),
                "C": ("cosinus, T = 1 000", wbd.schedule_betas("cosine", 1000)),
                "D": ("linéaire, T = 400, bornes par défaut", np.linspace(1e-4, 0.02, 400)),
                "S": ("linéaire, T = 400, bornes × 1000/T", wbd.schedule_betas("linear", 400))}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for (letter, (label, betas)), style in zip(SCHEDULES_15.items(), ("-", "-", "--", ":")):
    ab = wbd.alpha_bars(betas)
    frac = np.arange(1, len(ab) + 1) / len(ab)
    ax1.plot(frac, ab, style, lw=2, label=f"{letter} : {label}")
    ax2.plot(frac, np.log(ab / (1 - ab)), style, lw=2, label=letter)
ax1.set_xlabel("fraction de la chaîne, (indice + 1) / T")
ax1.set_ylabel("ᾱ")
ax1.legend(fontsize=8)
ax2.set_xlabel("fraction de la chaîne, (indice + 1) / T")
ax2.set_ylabel("ln SNR")
ax2.legend()
plt.show()

# One digit at the indices 0, 99, ..., 999 of the two schedules of T = 1 000, with the same noise
idx_15 = [0] + list(range(99, 1000, 100))
x0_15 = DIGITS.images[0].double().numpy()
eps_15 = np.random.default_rng(15).standard_normal(x0_15.shape)
grid_15 = [math.sqrt(a) * x0_15 + math.sqrt(1 - a) * eps_15
           for key in ("L", "C") for a in wbd.alpha_bars(SCHEDULES_15[key][1])[idx_15]]
show_images(np.stack(grid_15), titles=[f"{k} {i}" for k in ("L", "C") for i in idx_15], ncol=len(idx_15), size=1.35,
            suptitle="le même chiffre et le même bruit, aux indices 0 à 999 : linéaire (L), puis cosinus (C)")

In [ ]:
same_curve_15 = ...   # a) "D" or "S": which T = 400 schedule has the alpha_bar curve of L (against t / T)?
largest_end_15 = ...  # b) "L", "C", "D" or "S": which schedule keeps the most of the image at its last index?
cos_above_15 = ...    # c) "A", "B" or "C": over which share of the chain is ln SNR of C above that of L?
                      #    A: less than half; B: between half and 90 %; C: more than 95 %

In [ ]:
with wb.attempt("B5.15"):
    wb.check("B5.15a", same_curve_15)
with wb.attempt("B5.15"):
    wb.check("B5.15b", largest_end_15)
with wb.attempt("B5.15"):
    wb.check("B5.15c", cos_above_15)

### Ex B5.16 — q_posterior_mean_variance et predict_x0_from_eps 🔨 ★★ ⏱️ 30 min
**Objectif :** écrire la moyenne et la variance du posterior d'un pas, et l'estimation de $\mathbf{x}_0$ à partir d'un bruit.  
**Prérequis :** Ex B5.14 · ∂ B5.7 · fiche §205.3 (« Le posterior : un pas en arrière quand on connaît $\mathbf{x}_0$ », « Prédire le bruit ») · **mylearn :** `diffusion.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** : ouvre `mon_travail/mylearn/diffusion.py` (créé par `python tools/start_chapter.py B5`), lis la docstring, remplace chaque `raise NotImplementedError(...)` de l'exercice par ton code et **enregistre**. NumPy seulement. La première cellule de vérification recharge ta librairie et l'essaie sur des cas visibles ; la seconde lance les tests de l'exercice. `python -m pytest tests/test_chb5_diffusion.py -q` les lance tous dans un terminal.

**Contrat**
- `q_posterior_mean_variance(x0, x_t, t, schedule)` rend le couple `(mean, variance)` :
  - `schedule` est le dictionnaire de `noise_schedule` (B5.12) ; tu y lis `posterior_mean_coef1`, `posterior_mean_coef2` et `posterior_variance` à l'indice de chaque exemple ;
  - `mean` a la forme de `x0` ; `variance` a la forme `(N, 1, …, 1)`, avec **autant d'axes que `x0`** (`(N,)` si `x0` est 1-D), et vaut 0 à l'indice 0 ;
  - mêmes règles que `q_sample` pour `t` (un indice entier par exemple, dans $[0, T)$) ; `x_t` a **exactement** la forme de `x0`, sans diffusion.
- `predict_x0_from_eps(x_t, t, eps, alpha_bars)` : $\hat{\mathbf{x}}_0 = \big(\mathbf{x}_t - \sqrt{1 - \bar\alpha_t}\,\boldsymbol{\varepsilon}\big) / \sqrt{\bar\alpha_t}$, de la forme de `x_t` ; `eps` a exactement la forme de `x_t` ; mêmes règles pour `t`. L'aller-retour `predict_x0_from_eps(q_sample(x0, t, ab, noise)[0], t, noise, ab)` redonne `x0`.

La première cellule de vérification essaie tes deux fonctions sur les exemples des docstrings, les formes de la variance en 4-D et en 1-D, un **Bayes numérique** en dimension 1 (le produit des deux densités gaussiennes de ∂ B5.7, normalisé sur une grille de 600 001 points, dont on lit la moyenne et la variance), l'aller-retour avec le vrai bruit et quatre refus. La seconde lance les 29 tests de l'exercice (`-k "test_q_posterior_mean_variance_ or test_predict_x0_from_eps_"`).

Dans tes notes : à l'indice 0, que valent la moyenne et la variance du posterior, et pourquoi ? D'où vient l'écart entre la moyenne du Bayes numérique et ta formule, et que faudrait-il pour le réduire ? Que devient l'erreur de $\hat{\mathbf{x}}_0$ quand le bruit prédit se trompe de 0,1, au pas 10 puis au dernier pas du planning du notebook ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="B5")   # reload your saved file
with checked("B5.16"):
    M = mylearn.diffusion
    s_16 = toy_schedule([0.1, 0.2, 0.5])
    x0_16 = np.array([[1.0, -1.0], [0.5, 0.0]])
    xt_16 = np.array([[1.2649, -0.6325], [1.1, 0.8]])
    copies_16 = (x0_16.copy(), xt_16.copy())
    mean_16, var_16 = pair("B5.16", "q_posterior_mean_variance",
                           M.q_posterior_mean_variance(x0_16, xt_16, [0, 2], s_16), names=("mean", "variance"))
    t_16 = np.array([0, 2])
    same("B5.16", "moyenne du posterior (exemple de la docstring)", mean_16,
         s_16["posterior_mean_coef1"][t_16][:, None] * x0_16 + s_16["posterior_mean_coef2"][t_16][:, None] * xt_16,
         rtol=1e-6, atol=1e-6)
    same("B5.16", "variance du posterior, forme (2, 1)", var_16, s_16["posterior_variance"][t_16][:, None],
         rtol=1e-6, atol=1e-6)
    verdict("B5.16", np.asarray(var_16).ravel()[0] == 0, "à l'indice 0, la variance vaut 0 exactement.",
            "à l'indice 0, la variance du posterior doit valoir 0 exactement.")
    verdict("B5.16", all(np.array_equal(a, b) for a, b in zip((x0_16, xt_16), copies_16)),
            "les entrées n'ont pas changé.", "q_posterior_mean_variance a modifié x0 ou x_t.")
    _, var4_16 = M.q_posterior_mean_variance(np.zeros((3, 1, 2, 2)), np.ones((3, 1, 2, 2)), [0, 1, 2], s_16)
    verdict("B5.16", np.shape(var4_16) == (3, 1, 1, 1), "variance de forme (3, 1, 1, 1) pour x0 de forme (3, 1, 2, 2).",
            f"pour x0 de forme (3, 1, 2, 2), la variance doit avoir la forme (3, 1, 1, 1), reçu {np.shape(var4_16)}.")
    _, var1_16 = M.q_posterior_mean_variance(np.zeros(3), np.ones(3), [0, 1, 2], s_16)
    verdict("B5.16", np.shape(var1_16) == (3,), "variance de forme (3,) pour x0 en 1-D.",
            f"pour x0 de forme (3,), la variance doit avoir la forme (3,), reçu {np.shape(var1_16)}.")

    # Bayes on a fine grid, in 1-D: prior q(x_{t-1} | x0) times the likelihood q(x_t | x_{t-1}), at index 2
    x0_b, xt_b, t_b = -0.4, 0.7, 2
    grid = np.linspace(-15, 15, 600_001)
    ab, a = s_16["alpha_bars"], s_16["alphas"]
    log_post = (-(grid - math.sqrt(ab[t_b - 1]) * x0_b) ** 2 / (2 * (1 - ab[t_b - 1]))
                - (xt_b - math.sqrt(a[t_b]) * grid) ** 2 / (2 * (1 - a[t_b])))
    w_b = np.exp(log_post - log_post.max())
    m_b = float((grid * w_b).sum() / w_b.sum())
    mean_b, var_b = M.q_posterior_mean_variance(np.array([x0_b]), np.array([xt_b]), [t_b], s_16)
    same("B5.16", "moyenne contre Bayes sur une grille (x0 = −0,4, x_t = 0,7, indice 2)", mean_b, [m_b], rtol=1e-6, atol=1e-6)
    same("B5.16", "variance contre Bayes sur une grille", var_b, [float(((grid - m_b) ** 2 * w_b).sum() / w_b.sum())],
         rtol=1e-6, atol=1e-6)
    refuses("B5.16", "x_t de forme (2, 3) pour x0 de forme (2, 2)",
            lambda: M.q_posterior_mean_variance(x0_16, np.ones((2, 3)), [0, 1], s_16))
    refuses("B5.16", "t = 3 avec T = 3", lambda: M.q_posterior_mean_variance(x0_16, xt_16, [0, 3], s_16))

    eps_16 = np.array([[0.3, -2.0], [1.0, 0.5]])
    ab_16 = s_16["alpha_bars"]
    noisy_16 = np.sqrt(ab_16[[1, 2]])[:, None] * x0_16 + np.sqrt(1 - ab_16[[1, 2]])[:, None] * eps_16
    same("B5.16", "predict_x0_from_eps : l'aller-retour avec le vrai bruit redonne x0",
         M.predict_x0_from_eps(noisy_16, [1, 2], eps_16, ab_16), x0_16, rtol=1e-6, atol=1e-6)
    refuses("B5.16", "predict_x0_from_eps avec eps de forme (2,)",
            lambda: M.predict_x0_from_eps(noisy_16, [1, 2], np.ones(2), ab_16))
    refuses("B5.16", "predict_x0_from_eps avec t = -1", lambda: M.predict_x0_from_eps(noisy_16, [-1, 2], eps_16, ab_16))

In [ ]:
if written(lambda: mylearn.diffusion.predict_x0_from_eps([0.0], [0], [0.0], [0.5])):
    run_pytest_group(TEST_DIFF, "test_q_posterior_mean_variance_ or test_predict_x0_from_eps_")   # 29 tests
else:
    print("⏳ Ex B5.16 : les tests attendent tes fonctions.")

### Ex B5.17 — ddpm_step et ddim_step 🔨 ★★★ ⏱️ 40 min
**Objectif :** écrire un pas d'échantillonnage DDPM et un pas DDIM, avec leurs cas particuliers.  
**Prérequis :** Ex B5.16 · ✏️ B5.4, B5.5 · fiche §205.5 (« L'algorithme », « Un pas à la main », le piège « l'échelle et le dernier pas »), §205.6 (« L'idée de DDIM », « Le paramètre η ») · **mylearn :** `diffusion.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** : ouvre `mon_travail/mylearn/diffusion.py` (créé par `python tools/start_chapter.py B5`), lis la docstring, remplace chaque `raise NotImplementedError(...)` de l'exercice par ton code et **enregistre**. NumPy seulement. La première cellule de vérification recharge ta librairie et l'essaie sur des cas visibles ; la seconde lance les tests de l'exercice. `python -m pytest tests/test_chb5_diffusion.py -q` les lance tous dans un terminal.

**Contrat**
- `ddpm_step(x_t, t, eps_pred, schedule, rng=None, variance="posterior", clip_x0=None)` rend $\mathbf{x}_{t-1}$, de la forme de `x_t` :
  - `t` est un **entier Python** de $[0, T)$, le même pour tout le batch ; −1 est refusé ;
  - `variance` vaut exactement `"posterior"` ($\tilde\beta_t$) ou `"beta"` ($\beta_t$), et se vérifie **même à $t = 0$** : la clause *Raises* de la docstring est inconditionnelle ;
  - sans `clip_x0`, la moyenne de l'algorithme 2 de Ho et coll. ; avec `clip_x0`, $\hat{\mathbf{x}}_0$ écrêté à `[-clip_x0, clip_x0]` (la borne reçue, pas 1), puis la moyenne du posterior ;
  - à $t > 0$, on ajoute `sqrt(var) * z` : **l'écart-type**, pas la variance ; $\mathbf{z}$ est tiré de `rng`, de la forme de `x_t`, sans dépendre des valeurs de `x_t` ni de `eps_pred` (deux appels avec la même graine tirent le même $\mathbf{z}$ : c'est ainsi que les tests isolent la moyenne) ; **aucun bruit à $t = 0$**.
- `ddim_step(x_t, t, t_prev, eps_pred, alpha_bars, eta=0.0, rng=None)` rend $\mathbf{x}_{t_{\text{prev}}}$ :
  - `t_prev = -1` veut dire « l'image propre » : $\bar\alpha_{\text{prev}} = 1$ ; **ne lis pas `alpha_bars[-1]`** ;
  - `ValueError` si $-1 \le t_{\text{prev}} < t < T$ n'est pas vérifié (`t_prev >= t` refusé), ou si `eta` est hors de $[0, 1]$, `nan` compris ;
  - avec `eta = 0`, **aucun nombre n'est tiré de `rng`** (les tests vérifient l'état du générateur).

La première cellule de vérification essaie `ddpm_step` à $t = 0$ (l'exemple de la docstring), sa moyenne à l'indice 2 (deux appels à la même graine, l'un avec `x_t` et `eps_pred` nuls, dont la différence est la moyenne), le rapport des bruits des deux variances, l'écrêtage à 0,5 et trois refus ; puis `ddim_step` avec le vrai bruit (vers −1, puis vers l'indice 0), l'état de `rng` avec `eta = 0` et trois refus. La seconde lance les 101 tests de l'exercice (`-k "test_ddpm_step_ or test_ddim_step_"`).

Dans tes notes : pourquoi n'ajoute-t-on pas de bruit au dernier pas, et que verrais-tu sur des images si tu en ajoutais (fiche, piège « l'échelle et le dernier pas ») ? Pourquoi `alpha_bars[-1]` serait-il une erreur silencieuse pour `t_prev = -1` ? Avec `eta = 1` et `t_prev = t - 1`, que vaut $\sigma^2$, et qu'en conclus-tu sur le lien entre DDIM et DDPM ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="B5")   # reload your saved file
with checked("B5.17"):
    M = mylearn.diffusion
    s_17 = toy_schedule([0.1, 0.2, 0.5])
    b, a, ab = s_17["betas"], s_17["alphas"], s_17["alpha_bars"]

    def algo2_17(x, t, eps):        # the mean of Algorithm 2 (Ho et al.)
        return (np.asarray(x) - b[t] / math.sqrt(1 - ab[t]) * np.asarray(eps)) / math.sqrt(a[t])

    x_17, e_17 = np.array([[0.5, -0.5]]), np.array([[1.0, 1.0]])
    same("B5.17", "ddpm_step à t = 0 (exemple de la docstring : pas de bruit)", M.ddpm_step(x_17, 0, e_17, s_17),
         algo2_17(x_17, 0, e_17), rtol=1e-6, atol=1e-6)
    xb_17 = np.random.default_rng(1).standard_normal((4, 3))
    eb_17 = np.random.default_rng(2).standard_normal((4, 3))
    copies_17 = (xb_17.copy(), eb_17.copy())
    full_17 = M.ddpm_step(xb_17, 2, eb_17, s_17, rng=np.random.default_rng(7))
    again_17 = M.ddpm_step(xb_17, 2, eb_17, s_17, rng=np.random.default_rng(7))
    verdict("B5.17", np.array_equal(np.asarray(full_17), np.asarray(again_17)), "même graine, même résultat.",
            "deux appels avec la même graine doivent rendre le même x_{t-1} : z est-il tiré de rng ?")
    noise_17 = np.asarray(M.ddpm_step(np.zeros((4, 3)), 2, np.zeros((4, 3)), s_17, rng=np.random.default_rng(7)))
    same("B5.17", "moyenne à l'indice 2 (même graine : la différence de deux appels)", np.asarray(full_17) - noise_17,
         algo2_17(xb_17, 2, eb_17), rtol=1e-6, atol=1e-6)
    beta_17 = np.asarray(M.ddpm_step(np.zeros((4, 3)), 2, np.zeros((4, 3)), s_17, rng=np.random.default_rng(7),
                                     variance="beta"))
    same("B5.17", "bruit avec variance='beta' contre 'posterior' : le rapport des écarts-types",
         beta_17, noise_17 * math.sqrt(b[2] / s_17["posterior_variance"][2]), rtol=1e-6, atol=1e-6)
    verdict("B5.17", all(np.array_equal(u, v) for u, v in zip((xb_17, eb_17), copies_17)), "les entrées n'ont pas changé.",
            "ddpm_step a modifié x_t ou eps_pred.")
    big_17 = np.array([[-3.0, 2.0, 0.1]])                                     # x0_hat far outside [-0.5, 0.5]
    xc_17 = np.array([[0.2, -0.1, 0.3]])
    x0hat_17 = (xc_17 - math.sqrt(1 - ab[1]) * big_17) / math.sqrt(ab[1])
    clipped_17 = (np.asarray(M.ddpm_step(xc_17, 1, big_17, s_17, rng=np.random.default_rng(3), clip_x0=0.5))
                  - np.asarray(M.ddpm_step(np.zeros((1, 3)), 1, np.zeros((1, 3)), s_17, rng=np.random.default_rng(3),
                                           clip_x0=0.5)))
    same("B5.17", "clip_x0 = 0.5 : moyenne du posterior de x̂₀ écrêté à [−0,5 ; 0,5]", clipped_17,
         s_17["posterior_mean_coef1"][1] * np.clip(x0hat_17, -0.5, 0.5) + s_17["posterior_mean_coef2"][1] * xc_17,
         rtol=1e-6, atol=1e-6)
    refuses("B5.17", "variance='betas' (inconnue), même à t = 0", lambda: M.ddpm_step(x_17, 0, e_17, s_17, variance="betas"))
    refuses("B5.17", "ddpm_step à t = 3 avec T = 3", lambda: M.ddpm_step(x_17, 3, e_17, s_17))
    refuses("B5.17", "ddpm_step à t = -1", lambda: M.ddpm_step(x_17, -1, e_17, s_17))

    x0_17 = np.array([[0.5, -1.0, 0.25]])
    eps_17 = np.array([[1.0, 0.0, -2.0]])
    xt_17 = math.sqrt(ab[2]) * x0_17 + math.sqrt(1 - ab[2]) * eps_17
    same("B5.17", "ddim_step de l'indice 2 à -1 avec le vrai bruit : x0", M.ddim_step(xt_17, 2, -1, eps_17, ab), x0_17,
         rtol=1e-6, atol=1e-6)
    same("B5.17", "ddim_step de l'indice 2 à 0 avec le vrai bruit : le x au pas 0 du même bruit",
         M.ddim_step(xt_17, 2, 0, eps_17, ab), math.sqrt(ab[0]) * x0_17 + math.sqrt(1 - ab[0]) * eps_17,
         rtol=1e-6, atol=1e-6)
    rng_17 = np.random.default_rng(11)
    state_17 = copy.deepcopy(rng_17.bit_generator.state)
    M.ddim_step(xt_17, 2, 1, eps_17, ab, eta=0.0, rng=rng_17)
    verdict("B5.17", rng_17.bit_generator.state == state_17, "avec eta = 0, rng n'a pas servi.",
            "avec eta = 0, ddim_step ne doit tirer aucun nombre de rng (son état a changé).")
    refuses("B5.17", "ddim_step avec t_prev = t", lambda: M.ddim_step(xt_17, 2, 2, eps_17, ab))
    refuses("B5.17", "ddim_step avec eta = 1.5", lambda: M.ddim_step(xt_17, 2, 1, eps_17, ab, eta=1.5))
    refuses("B5.17", "ddim_step avec eta = nan", lambda: M.ddim_step(xt_17, 2, 1, eps_17, ab, eta=float("nan")))

In [ ]:
if written(lambda: mylearn.diffusion.ddim_step([0.0], 1, 0, [0.0], [0.9, 0.8])):
    run_pytest_group(TEST_DIFF, "test_ddpm_step_ or test_ddim_step_")   # 101 tests
else:
    print("⏳ Ex B5.17 : les tests attendent tes fonctions.")

### Ex B5.18 — Avec le vrai bruit, DDPM et DDIM retrouvent-ils x_0 ? Contrôle sur des données 2D 🔬 ★★ ⏱️ 25 min
**Objectif :** enchaîner tes pas DDPM et DDIM avec le vrai bruit, pour savoir lesquels doivent retrouver $\mathbf{x}_0$ exactement : le contrôle de ta librairie avant tout entraînement.  
**Prérequis :** Ex B5.17 · fiche §205.5 (« L'algorithme »), §205.6 (« L'idée de DDIM », « Le paramètre η ») · **Fil rouge :** synthétique · **Parcours :** M, C

Un réseau parfait prédirait le **vrai** bruit. Sans réseau, on peut donc éprouver une chaîne d'échantillonnage : partir d'une donnée $\mathbf{x}_0$ connue, la bruiter jusqu'au dernier indice, puis remonter en donnant aux pas « le vrai bruit » à la place de $\boldsymbol{\varepsilon}_\theta$. La fiche dit qu'**un** pas DDIM vers l'indice −1 rend alors $\mathbf{x}_0$. Mais quel est « le vrai bruit » au milieu d'une chaîne, et qu'en est-il des chaînes entières ?

**Contrat**
- `true_noise_18(x_t, t, x0, alpha_bars)` : le bruit $\boldsymbol{\varepsilon}$ tel que $\mathbf{x}_t = \sqrt{\bar\alpha_t}\,\mathbf{x}_0 + \sqrt{1 - \bar\alpha_t}\,\boldsymbol{\varepsilon}$, pour un indice `t` entier (le même pour tout le batch) ; un nouveau tableau, de la forme de `x_t` ;
- `ddpm_chain_18(x0, eps, schedule, rng, oracle=False)` : $\mathbf{x}$ à l'indice $T - 1$ fait de `x0` et `eps` par la forme fermée, puis ta `ddpm_step` de l'indice $T - 1$ à 0 (avec `rng`, sans écrêtage, la variance par défaut), avec pour `eps_pred` le bruit **initial** `eps` à chaque pas (`oracle=False`), ou le vrai bruit du $\mathbf{x}$ **courant**, `true_noise_18(x, t, x0, …)` (`oracle=True`) ; rend le $\mathbf{x}$ final ;
- `ddim_chain_18(x0, eps, alpha_bars, timesteps, eta=0.0, rng=None)` : $\mathbf{x}$ à l'indice `timesteps[0]` fait de `x0` et `eps`, puis ta `ddim_step` de chaque indice de la sous-séquence décroissante `timesteps` au suivant, et du dernier à −1, toujours avec `eps_pred = eps`, le bruit initial ; rend le $\mathbf{x}$ final.

La cellule de vérification essaie tes trois fonctions sur un planning de 3 pas, contre la chaîne écrite pas à pas. Puis l'expérience prend 500 points de l'anneau de 8 gaussiennes (`wbd.ring`), le planning du notebook (ta `noise_schedule` de `BETAS`), et mesure l'écart maximal entre $\mathbf{x}_0$ et la fin de cinq chaînes : DDPM avec le bruit initial ; DDPM avec le vrai bruit du $\mathbf{x}$ courant ; DDIM ($\eta = 0$) sur tous les indices ; DDIM ($\eta = 0$) sur 10 indices (`wbd.ddim_timesteps(T_STEPS, 10)`) ; DDIM ($\eta = 1$) sur tous les indices. Réponds ensuite, d'après le tableau, par `True` ou `False` : la chaîne retrouve-t-elle $\mathbf{x}_0$ à $10^{-6}$ près ?
- a) `ddpm_fixed_18` : DDPM, bruit initial à chaque pas ;
- b) `ddpm_oracle_18` : DDPM, vrai bruit du $\mathbf{x}$ courant ;
- c) `ddim_ten_18` : DDIM, $\eta = 0$, 10 pas ;
- d) `ddim_eta1_18` : DDIM, $\eta = 1$, tous les pas.

Dans tes notes : pourquoi le bruit initial n'est-il plus « le vrai bruit » d'un $\mathbf{x}_t$ de la chaîne DDPM après le premier pas, alors qu'il le reste tout au long de la chaîne DDIM avec $\eta = 0$ ? Pourquoi la chaîne DDPM avec le vrai bruit courant finit-elle exactement sur $\mathbf{x}_0$, quelle que soit sa trajectoire (regarde le dernier pas) ? Pourquoi le nombre de pas de DDIM ne change-t-il rien ici, alors qu'il comptera avec un réseau ? Quelle erreur de ta librairie ce contrôle aurait-il attrapée (un indice décalé, $\bar\alpha$ au lieu de $\sqrt{\bar\alpha}$), et laquelle pas ?

In [ ]:
def true_noise_18(x_t, t, x0, alpha_bars):
    """The noise eps such that x_t = sqrt(alpha_bar_t) x0 + sqrt(1 - alpha_bar_t) eps, for one step index t
    (an int, the same for the whole batch); a new array with the shape of x_t."""
    raise NotImplementedError("true_noise_18")


def ddpm_chain_18(x0, eps, schedule, rng, oracle=False):
    """The whole DDPM chain: x at index T - 1 made from x0 and eps by the closed form, then ddpm_step (your library)
    from index T - 1 down to 0, with as eps_pred the initial eps (oracle=False) or the true noise of the current x
    (true_noise_18, oracle=True). Returns the final x."""
    raise NotImplementedError("ddpm_chain_18")


def ddim_chain_18(x0, eps, alpha_bars, timesteps, eta=0.0, rng=None):
    """The DDIM chain on the decreasing sub-sequence timesteps: x at index timesteps[0] made from x0 and eps by the
    closed form, then ddim_step (your library) from each index to the next one, and from the last one to -1, always
    with eps_pred = eps, the initial noise. Returns the final x."""
    raise NotImplementedError("ddim_chain_18")


ddpm_fixed_18 = ...    # a) DDPM, initial eps at every step: does it return x0 within 1e-6? True or False
ddpm_oracle_18 = ...   # b) DDPM, true noise of the current x at every step: within 1e-6 of x0? True or False
ddim_ten_18 = ...      # c) DDIM, eta = 0, 10 steps, initial eps: within 1e-6 of x0? True or False
ddim_eta1_18 = ...     # d) DDIM, eta = 1, all T steps, initial eps: within 1e-6 of x0? True or False

In [ ]:
chains_ok_18 = False
with checked("B5.18"):
    s_18 = toy_schedule([0.1, 0.2, 0.5])
    ab_18 = s_18["alpha_bars"]
    x0_c = np.array([[0.3, -0.7], [1.2, 0.1]])
    e_c = np.array([[0.5, 1.0], [-1.5, 0.2]])
    xt_c = math.sqrt(ab_18[2]) * x0_c + math.sqrt(1 - ab_18[2]) * e_c
    copy_c = xt_c.copy()
    ok_18 = same("B5.18", "true_noise_18 redonne le bruit de x_t", true_noise_18(xt_c, 2, x0_c, ab_18), e_c,
                 rtol=1e-9, atol=1e-12)
    verdict("B5.18", np.array_equal(xt_c, copy_c), "true_noise_18 n'a pas modifié x_t.", "true_noise_18 a modifié x_t.")
    ok_18 &= np.array_equal(xt_c, copy_c)
    # the DDPM chain with T = 3 and the initial noise, written step by step without your functions
    rng_a, rng_b = np.random.default_rng(4), np.random.default_rng(4)
    x_ref = xt_c.copy()
    for t in (2, 1, 0):
        mean = (x_ref - s_18["betas"][t] / math.sqrt(1 - ab_18[t]) * e_c) / math.sqrt(s_18["alphas"][t])
        x_ref = mean if t == 0 else mean + np.asarray(mylearn.diffusion.ddpm_step(
            np.zeros_like(x_ref), t, np.zeros_like(x_ref), s_18, rng=rng_b))
    ok_18 &= same("B5.18", "ddpm_chain_18 (T = 3, bruit initial) contre la chaîne pas à pas",
                  ddpm_chain_18(x0_c, e_c, s_18, rng_a), x_ref, rtol=1e-6, atol=1e-6)
    x_dd = math.sqrt(ab_18[2]) * x0_c + math.sqrt(1 - ab_18[2]) * e_c
    x0h = (x_dd - math.sqrt(1 - ab_18[2]) * e_c) / math.sqrt(ab_18[2])
    ok_18 &= same("B5.18", "ddim_chain_18 sur [2, 0] (η = 0) contre la formule", ddim_chain_18(x0_c, e_c, ab_18, [2, 0]),
                  x0h, rtol=1e-6, atol=1e-6)
    chains_ok_18 = bool(ok_18)
if not chains_ok_18:
    print("⏳ Ex B5.18 : l'expérience attend tes trois fonctions (et ddpm_step, ddim_step de B5.17).")
else:
    X0_18 = wbd.ring(500, seed=18)                                   # 500 points of the ring of 8 Gaussians
    EPS_18 = np.random.default_rng(180).standard_normal(X0_18.shape)
    SCHED_18 = mylearn.diffusion.noise_schedule(BETAS)
    AB_18 = SCHED_18["alpha_bars"]
    runs_18 = {"DDPM, bruit initial à chaque pas": ddpm_chain_18(X0_18, EPS_18, SCHED_18, np.random.default_rng(1)),
               "DDPM, vrai bruit du x courant": ddpm_chain_18(X0_18, EPS_18, SCHED_18, np.random.default_rng(1), oracle=True),
               f"DDIM η = 0, les {T_STEPS} pas": ddim_chain_18(X0_18, EPS_18, AB_18, np.arange(T_STEPS - 1, -1, -1)),
               "DDIM η = 0, 10 pas": ddim_chain_18(X0_18, EPS_18, AB_18, wbd.ddim_timesteps(T_STEPS, 10)),
               f"DDIM η = 1, les {T_STEPS} pas": ddim_chain_18(X0_18, EPS_18, AB_18, np.arange(T_STEPS - 1, -1, -1),
                                                                eta=1.0, rng=np.random.default_rng(2))}
    ERRORS_18 = {name: float(np.abs(np.asarray(x) - X0_18).max()) for name, x in runs_18.items()}
    for name, err in ERRORS_18.items():
        print(f"{name:>32} : écart maximal à x0 = {fr_e(err)}")
    verdict("B5.18", max(ERRORS_18[k] for k in list(ERRORS_18)[1:4]) < 1e-9,
            "les trois chaînes exactes retrouvent x0 à moins de 1e-9.",
            "une chaîne qui devrait être exacte s'écarte de x0 de plus de 1e-9 : relis tes trois fonctions.")
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))         # each with its own scale
    for ax, name in zip(axes, [list(runs_18)[0], list(runs_18)[3], list(runs_18)[4]]):
        ax.scatter(*X0_18.T, s=4, alpha=0.4, label="x0")
        ax.scatter(*np.asarray(runs_18[name]).T, s=4, alpha=0.4, label="fin de la chaîne")
        ax.set_title(name, fontsize=9)
        ax.set_aspect("equal")
    axes[0].legend(fontsize=8)
    plt.show()
    for letter, answer in zip("abcd", [ddpm_fixed_18, ddpm_oracle_18, ddim_ten_18, ddim_eta1_18]):
        with wb.attempt("B5.18"):
            wb.check(f"B5.18{letter}", answer)

## Partie B · Jouets 2D : un MLP débruiteur, puis le flow matching

Avant les images, des points du plan : l'**anneau de 8 gaussiennes** (`wbd.ring` : 8 centres sur un cercle de rayon 1,5, écart-type 0,15 autour de chacun ; sa variance par coordonnée, 1,15, est proche de celle du bruit). Tout s'y voit d'un coup d'œil et s'entraîne en une dizaine de secondes. Tu y entraînes un MLP qui prédit le bruit, conditionné par le pas comme le U-Net de la fiche, puis tu l'échantillonnes avec ta `ddpm_step` (B5.19) ; puis tu entraînes le **même** MLP à prédire une vitesse (flow matching, fiche §205.9) et tu compares les deux échantillonneurs à nombre de pas égal (B5.20).

**Mesurer la qualité.** Une distance de Fréchet en 2D ne compare que les moyennes et les covariances : un nuage gaussien de variance 1 a presque celles de l'anneau. On prend la **distance de Wasserstein en tranches** (`wbd.sliced_wasserstein`) : la moyenne, sur 64 directions du plan, de la distance de Wasserstein du ch. 27 entre les deux projections (l'écart moyen de leurs valeurs triées). La fonction `quality(points)` rend cette distance à `REAL_B` (2 000 points de l'anneau), la part des points à moins de trois écarts-types d'un centre et le nombre de modes atteints (`wbd.mode_report`) ; la cellule suivante affiche ses deux repères, deux échantillons de l'anneau et du bruit pur.

**Encoder le pas.** Le MLP reçoit une **position entière**, comme le U-Net : l'indice $t$ du pas pour la diffusion ; pour le flow matching, dont le temps $s$ est continu sur $[0, 1]$, on le prend sur une grille de 1 000 valeurs, $s = u / 1\,000$ avec $u$ entier de 1 à 1 000 (`wbd.FM_POSITIONS` = 1 001 positions). La position $p$ est encodée par la ligne $p$ de ta `positional_encoding` de B3 (`mylearn.attention`) : si tu n'as pas écrit `attention.py`, la référence la remplace (une ligne ℹ️ le dit) ; s'il existe sans cette fonction, la vérification te le signale.

`SETTINGS_B` donne le budget des deux entraînements : 3 000 itérations, des batches de 512 points, AdamW au learning rate 2e-3 (environ 10 s chacun sur un CPU à deux cœurs). `wbd` contient une solution de référence (`Denoiser2D`, `train_toy`, `sample_toy_ddpm`…) : la vérification s'en sert pour comparer tes fonctions ; ne la lis qu'une fois les tiennes écrites.

In [ ]:
# Part B: the ring of 8 Gaussians, the settings of the two trainings, the measures of quality
DATA_B = torch.tensor(wbd.ring(50_000, seed=1900), dtype=torch.float32)   # 50 000 training points
REAL_B = wbd.ring(2000, seed=2020)                                         # 2 000 other points, for the measures
SETTINGS_B = dict(wbd.TOY_SETTINGS)                    # 3 000 iterations, batches of 512, AdamW at 2e-3 (≈ 10 s)
SCHED_B = toy_schedule(BETAS)                          # the schedule of the notebook, T = T_STEPS (without your code)
FLOOR_B = wbd.sliced_wasserstein(wbd.ring(2000, seed=2021), REAL_B)        # two samples of the ring itself
BLOB_B = wbd.sliced_wasserstein(np.random.default_rng(3).standard_normal((2000, 2)), REAL_B)   # pure noise


def quality(points):
    """(sliced Wasserstein distance to REAL_B, share of points on a mode, modes reached) of 2 000 points."""
    report = wbd.mode_report(points)
    return wbd.sliced_wasserstein(points, REAL_B), report["on_mode"], report["modes"]


print(f"distance de Wasserstein en tranches : {fr(FLOOR_B, 3)} entre deux échantillons de l'anneau, "
      f"{fr(BLOB_B, 3)} pour du bruit gaussien.")

### Ex B5.19 — Diffusion en 2D : un MLP débruiteur conditionné par le pas de temps, entraîné puis échantillonné 📦 ★★ ⏱️ 30 min
**Objectif :** entraîner un MLP à prédire le bruit de points 2D, puis générer de nouveaux points avec ta `ddpm_step`.  
**Prérequis :** Ex B5.17 · B3 (`positional_encoding`, B3.15) · fiche §205.3 (« La loss simplifiée »), §205.4 (« Dire au réseau quel est le pas »), §205.5 (« L'algorithme ») · **Fil rouge :** synthétique · **Parcours :** R, M, C

**Contrat**
- `Denoiser2D_19(n_positions, t_dim=32, hidden=128)`, un `nn.Module` :
  - un buffer **non persistant** `pe` (`register_buffer(..., persistent=False)` : il n'entre pas dans le `state_dict`), la table `positional_encoding(n_positions, t_dim)` de ta librairie (B3) en float32 ;
  - `net = nn.Sequential(nn.Linear(2 + t_dim, hidden), nn.SiLU(), nn.Linear(hidden, hidden), nn.SiLU(), nn.Linear(hidden, hidden), nn.SiLU(), nn.Linear(hidden, 2))` ;
  - `forward(x, t)` : `x` de forme `(N, 2)` en float32, `t` de forme `(N,)` en int64 → `net` appliqué aux coordonnées **suivies** de l'encodage de la position, `(N, 2)`.
- `train_19(model, data, alpha_bars, n_iters, batch_size, lr, generator)` entraîne `model` sur place et rend la liste des `n_iters` loss (des floats Python) :
  - un `torch.optim.AdamW(model.parameters(), lr=lr)`, avec ses autres réglages par défaut ;
  - à chaque itération, **dans cet ordre**, trois tirages par `generator` : les indices du batch `torch.randint(len(data), (batch_size,))`, les pas `torch.randint(0, T, (batch_size,))` ($T$ = `len(alpha_bars)`), le bruit `torch.randn((batch_size, 2))` ; puis $\mathbf{x}_t$ par la forme fermée en float32, la loss `F.mse_loss(model(x_t, t), eps)`, les gradients et un pas de l'optimiseur.
- `sample_19(model, n, schedule, generator, rng)` : `n` points par l'échantillonnage ancestral : $\mathbf{x}$ = `torch.randn((n, 2), generator=generator)` converti en float64 NumPy, puis, pour $t$ de $T - 1$ à 0, la prédiction du réseau (sans gradient, en float32) et ta `ddpm_step(x, t, eps, schedule, rng=rng)` (variance du posterior, sans écrêtage) ; rend un tableau float64 `(n, 2)`.

La cellule de vérification contrôle ton MLP (noms et formes du `state_dict`, le buffer `pe`, la sortie aux mêmes poids que le modèle du contrat), 5 itérations de `train_19` contre la recette du contrat (mêmes tirages, mêmes loss) et `sample_19` sur 6 points. La cellule suivante entraîne ton MLP avec les réglages de `SETTINGS_B` sur `DATA_B`, génère 2 000 points, trace la loss et les points, et vérifie qu'ils couvrent les 8 modes (une distance en tranches sous 0,2, plus de 80 % des points sur un mode). Le temps affiché est une mesure, pas une réponse.

Dans tes notes : la loss descend-elle vers 0 ? Pourquoi pas, alors que le modèle génère de bons points (pense à ce que vaut l'erreur sur $\boldsymbol{\varepsilon}$ aux petits indices, où $\mathbf{x}_t$ ne dit presque rien du bruit) ? Combien de passes du réseau a coûté l'échantillonnage ? Que se passe-t-il si tu supprimes l'encodage du pas (des zéros à la place) ?

In [ ]:
class Denoiser2D_19(nn.Module):
    """MLP that predicts the noise of a 2-D point x_t at the integer position t (contract of the statement)."""

    def __init__(self, n_positions, t_dim=32, hidden=128):
        super().__init__()
        raise NotImplementedError("Denoiser2D_19")

    def forward(self, x, t):
        raise NotImplementedError("Denoiser2D_19.forward")


def train_19(model, data, alpha_bars, n_iters, batch_size, lr, generator):
    """Train model to predict the noise (recipe of the contract); returns the list of the losses (floats)."""
    raise NotImplementedError("train_19")


def sample_19(model, n, schedule, generator, rng):
    """n points by DDPM ancestral sampling with your ddpm_step; a float64 array (n, 2)."""
    raise NotImplementedError("sample_19")

In [ ]:
model_ok_19 = False
with checked("B5.19"):
    torch.manual_seed(0)
    mine_19 = Denoiser2D_19(50)
    ref_19 = wbd.Denoiser2D(50)
    names_19 = {k: tuple(v.shape) for k, v in mine_19.state_dict().items()}
    want_19 = {k: tuple(v.shape) for k, v in ref_19.state_dict().items()}
    ok_19 = names_19 == want_19
    verdict("B5.19", ok_19, "noms et formes des paramètres : ceux du contrat (net.0 à net.6).",
            f"le state_dict doit avoir les noms et formes {want_19} ; reçu {names_19} (le buffer pe est-il non persistant ?).")
    if not ok_19:
        raise StopCheck
    pe_19 = getattr(mine_19, "pe", None)
    pe_ok_19 = (pe_19 is not None and tuple(pe_19.shape) == (50, 32)
                and np.allclose(pe_19.double().numpy(), wbd.sinusoidal_table(50, 32), atol=1e-6))
    verdict("B5.19", pe_ok_19, "le buffer pe est la table de positional_encoding(50, 32) (B3).",
            "self.pe doit être la table positional_encoding(n_positions, t_dim) de B3, en float32.")
    ok_19 &= pe_ok_19
    with torch.no_grad():
        for p_ in ref_19.parameters():
            p_.add_(0.3 * torch.randn(p_.shape))                    # weights far from the initialisation
    mine_19.load_state_dict(ref_19.state_dict())
    xq_19, tq_19 = torch.randn(7, 2), torch.tensor([0, 3, 9, 17, 25, 33, 49])
    with torch.no_grad():
        ok_19 &= same("B5.19", "sortie de ton MLP contre celle du contrat, aux mêmes poids", mine_19(xq_19, tq_19),
                      ref_19(xq_19, tq_19), rtol=1e-5, atol=1e-6)
    # 5 iterations of train_19 against the recipe of the contract (the same draws, the same model at the start)
    ab_19 = toy_schedule(np.linspace(1e-3, 0.2, 50))["alpha_bars"]
    a_19, b_19 = copy.deepcopy(ref_19), copy.deepcopy(ref_19)
    got_19 = train_19(a_19, DATA_B, ab_19, 5, 64, 1e-3, torch.Generator().manual_seed(5))
    want_l19 = wbd.train_toy(b_19, DATA_B, 5, 64, 1e-3, torch.Generator().manual_seed(5), ab_19)
    if returned("B5.19", "train_19", got_19):
        ok_19 &= same("B5.19", "les 5 premières loss de train_19 contre la recette du contrat (mêmes tirages, dans le même ordre)", got_19, want_l19,
                      rtol=1e-5, atol=1e-6)
        ok_19 &= same("B5.19", "les poids après 5 itérations", a_19.net[0].weight, b_19.net[0].weight, rtol=1e-4,
                      atol=1e-6)
    else:
        ok_19 = False
    sched_19 = toy_schedule(np.linspace(1e-3, 0.2, 50))
    b_19.eval()
    pts_19 = sample_19(b_19, 6, sched_19, torch.Generator().manual_seed(8), np.random.default_rng(9))
    if returned("B5.19", "sample_19", pts_19):
        ok_19 &= same("B5.19", "sample_19 (6 points, T = 50) contre l'échantillonnage DDPM du contrat", pts_19,
                      wbd.sample_toy_ddpm(b_19, 6, sched_19["alpha_bars"], torch.Generator().manual_seed(8),
                                          np.random.default_rng(9)), rtol=1e-5, atol=1e-5)
        model_ok_19 = bool(ok_19)

In [ ]:
# Train on the ring, then 2 000 points by DDPM (≈ 10 s of training)
MODEL_19 = None
if not model_ok_19:
    print("⏳ Ex B5.19 : l'entraînement attend ton MLP, train_19 et sample_19 (cellule précédente sans ❌).")
else:
    torch.manual_seed(19)
    MODEL_19 = Denoiser2D_19(T_STEPS)
    start_19 = time.perf_counter()
    LOSSES_19 = train_19(MODEL_19, DATA_B, SCHED_B["alpha_bars"], SETTINGS_B["n_iters"], SETTINGS_B["batch_size"],
                         SETTINGS_B["lr"], torch.Generator().manual_seed(190))
    MODEL_19.eval()
    POINTS_19 = sample_19(MODEL_19, 2000, SCHED_B, torch.Generator().manual_seed(191), np.random.default_rng(192))
    swd_19, on_19, modes_19 = quality(POINTS_19)
    print(f"entraînement et échantillonnage : {time.perf_counter() - start_19:.0f} s (une mesure, pas une réponse)")
    print(f"loss moyenne des 200 dernières itérations : {fr(np.mean(LOSSES_19[-200:]), 3)} ; distance en tranches "
          f"{fr(swd_19, 3)} ; {fr(100 * on_19, 1)} % des points sur un mode ; {modes_19} modes sur 8")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(np.convolve(LOSSES_19, np.ones(50) / 50, mode="valid"))
    ax1.set_xlabel("itération")
    ax1.set_ylabel("loss (moyenne glissante sur 50)")
    ax2.scatter(*REAL_B.T, s=3, alpha=0.3, label="anneau")
    ax2.scatter(*POINTS_19.T, s=3, alpha=0.3, label="DDPM")
    ax2.set_aspect("equal")
    ax2.legend()
    plt.show()
    verdict("B5.19", swd_19 < 0.2 and on_19 > 0.8 and modes_19 == 8,
            "les points générés couvrent les 8 modes, à distance de l'anneau proche de celle de deux échantillons.",
            "les points générés sont loin de l'anneau (distance ≥ 0,2, moins de 80 % sur un mode, ou un mode manqué) : "
            "relis l'entraînement (le pas t donné au modèle, la cible) et l'échantillonnage.")

### Ex B5.20 — Flow matching en 2D : même MLP, cible « vitesse » et intégration d'Euler 🔬 ★★★ ⏱️ 40 min
**Objectif :** entraîner le même MLP à prédire une vitesse, l'intégrer par la méthode d'Euler, et comparer la qualité au DDIM du modèle de B5.19 selon le nombre de pas, sans présumer du gagnant.  
**Prérequis :** Ex B5.19 · fiche §205.9 (« Flow matching et rectified flow », encadré « la méthode d'Euler »), §205.6 (« L'idée de DDIM ») · **Fil rouge :** synthétique · **Parcours :** M, C

Le flow matching relie la donnée et le bruit par la ligne droite $\mathbf{x}_s = (1 - s)\,\mathbf{x}_0 + s\,\boldsymbol{\varepsilon}$ (fiche §205.9) et fait prédire au réseau la vitesse $\mathbf{v} = \boldsymbol{\varepsilon} - \mathbf{x}_0$ ; on génère en suivant le champ des vitesses de $s = 1$ (le bruit) à $s = 0$. La question : à nombre de pas égal, ces points valent-ils mieux que ceux de DDIM avec le modèle de B5.19, qui prédit le bruit ? Même MLP, même budget d'entraînement, mêmes points de départ.

**Contrat**
- `train_fm_20(model, data, n_iters, batch_size, lr, generator)` : la recette de `train_19`, à trois différences près : le deuxième tirage est la position `u = torch.randint(1, 1001, (batch_size,))`, et $s = u / 1\,000$ ; l'entrée est $\mathbf{x}_s$ ; la cible est $\boldsymbol{\varepsilon} - \mathbf{x}_0$, et le modèle reçoit la position `u` (un `Denoiser2D_19(wbd.FM_POSITIONS)`, 1 001 positions). Rend la liste des loss.
- `euler_20(model, n, n_steps, generator)` : $\mathbf{x}$ = `torch.randn((n, 2), generator=generator)`, puis `n_steps` pas d'Euler égaux, de $s = 1$ vers $s = 0$ : au pas $k$ (de 0 à `n_steps` − 1), la position `round(1000 * (1 - k / n_steps))` et $\mathbf{x} \leftarrow \mathbf{x} - \mathbf{v}/\text{n\_steps}$ ; sans gradient ; rend un tableau float64 `(n, 2)`.
- `ddim_20(model, n, n_steps, alpha_bars, generator)` : $\mathbf{x}$ = `torch.randn((n, 2), generator=generator)` en float64 NumPy, puis ta `ddim_step` ($\eta = 0$) d'un indice au suivant de `wbd.ddim_timesteps(T, n_steps)` (de $T - 1$ à 0, `n_steps` indices), et du dernier à −1 ; rend un tableau float64 `(n, 2)`.

La cellule de vérification compare tes trois fonctions aux fonctions du contrat (5 itérations, puis 1, 3 et 8 pas d'Euler, 1, 4 et 50 pas de DDIM). La cellule suivante (environ 15 s) entraîne le modèle de flow matching avec `SETTINGS_B`, puis mesure la distance en tranches de 2 000 points de chaque échantillonneur pour chaque nombre de pas de `STEPS_20`, avec le modèle de B5.19 pour DDIM ; elle trace les deux courbes, avec, en bandes, les **cinq graines** du protocole complet (`FULL_20`, précalculé avec les mêmes cellules ; ta graine est la première), et montre les points à 5 pas. Réponds ensuite d'après le tableau du protocole complet, **pour toutes ses graines** :
- a) `fewest_20` : le plus petit nombre de pas de `STEPS_20` à partir duquel la distance d'Euler est au plus le triple de sa distance à 100 pas ;
- b) `better5_20` : à 5 pas, lequel a la plus petite distance : `"D"` (DDIM) ou `"E"` (Euler) ?

Dans tes notes : pourquoi DDIM à 1 ou 2 pas rend-il des points aussi loin (regarde $\hat{\mathbf{x}}_0$ au dernier indice, et la division par $\sqrt{\bar\alpha_t}$) ? Et Euler à 1 pas, où envoie-t-il tous les points (fiche, encadré 🧮 « la MSE apprend une moyenne ») ? Le gagnant change-t-il avec le nombre de pas ? À 100 pas, compare le DDIM et le DDPM de B5.19 sur le même modèle : que conclus-tu de l'aléa de l'échantillonnage ancestral ? Qu'est-ce que la comparaison ne dit pas (un seul dataset, des réglages choisis pour l'un ou l'autre, la loss de chaque modèle) ?

In [ ]:
STEPS_20 = (1, 2, 5, 10, 20, 50, 100)                       # numbers of sampling steps of the comparison
# The full protocol (tools/chapters/history_chB5.py): the same cells with 5 seeds (seed 0 is yours), the
# distance of 2 000 points to REAL_B for each number of steps; DDIM depends on T, so on the mode
FULL_20 = {
    "seeds": [0, 1, 2, 3, 4],
    "DDIM (bruit prédit, B5.19)": [[6.8056, 6.6697, 0.4236, 0.2481, 0.1714, 0.147, 0.1404], [6.1998, 6.0638, 0.2993, 0.166, 0.1385, 0.1305, 0.1267], [5.4068, 5.2936, 0.2236, 0.1235, 0.0988, 0.0944, 0.0941], [6.4142, 6.281, 0.4477, 0.2235, 0.142, 0.1142, 0.1103], [6.4573, 6.3336, 0.2506, 0.1266, 0.0929, 0.0793, 0.078]] if FAST_MODE else [[3.3798, 3.3267, 0.4504, 0.2462, 0.1769, 0.1459, 0.1325], [5.2962, 5.2241, 0.3061, 0.1471, 0.1043, 0.0893, 0.0885], [7.4443, 7.3397, 0.4981, 0.1764, 0.1343, 0.1246, 0.1249], [4.9974, 4.9374, 0.3049, 0.1222, 0.0803, 0.0845, 0.0825], [5.2451, 5.1807, 0.2499, 0.0878, 0.09, 0.0913, 0.0908]],
    "Euler (vitesse prédite)": [[0.9146, 0.3714, 0.1034, 0.0814, 0.0774, 0.0751, 0.0749], [0.8998, 0.3423, 0.0725, 0.0441, 0.0403, 0.0414, 0.0404], [0.8987, 0.3947, 0.0889, 0.0574, 0.0502, 0.0509, 0.0505], [0.8727, 0.3712, 0.1305, 0.1156, 0.109, 0.1006, 0.1009], [0.8969, 0.3872, 0.084, 0.0549, 0.0442, 0.0393, 0.0361]],
}

In [ ]:
def train_fm_20(model, data, n_iters, batch_size, lr, generator):
    """Train model to predict the velocity eps - x0 at s = u / 1000 (recipe of the contract); returns the losses."""
    raise NotImplementedError("train_fm_20")


def euler_20(model, n, n_steps, generator):
    """n points by Euler integration of the flow from s = 1 to s = 0 in n_steps steps; a float64 array (n, 2)."""
    raise NotImplementedError("euler_20")


def ddim_20(model, n, n_steps, alpha_bars, generator):
    """n points by DDIM (eta = 0, your ddim_step) on wbd.ddim_timesteps(T, n_steps), then to -1; float64 (n, 2)."""
    raise NotImplementedError("ddim_20")

In [ ]:
fm_ok_20 = False
with checked("B5.20"):
    torch.manual_seed(1)
    ref_20 = wbd.Denoiser2D(wbd.FM_POSITIONS)
    a_20, b_20 = copy.deepcopy(ref_20), copy.deepcopy(ref_20)
    got_20 = train_fm_20(a_20, DATA_B, 5, 64, 1e-3, torch.Generator().manual_seed(6))
    ok_20 = returned("B5.20", "train_fm_20", got_20)
    if ok_20:
        ok_20 = same("B5.20", "les 5 premières loss de train_fm_20 contre la recette du contrat (mêmes tirages, dans le même ordre)", got_20,
                     wbd.train_toy(b_20, DATA_B, 5, 64, 1e-3, torch.Generator().manual_seed(6), flow=True),
                     rtol=1e-5, atol=1e-6)
    b_20.eval()
    for steps_c in (1, 3, 8):
        ok_20 &= same("B5.20", f"euler_20 en {steps_c} pas contre l'intégration du contrat",
                      euler_20(b_20, 5, steps_c, torch.Generator().manual_seed(7)),
                      wbd.sample_toy_euler(b_20, 5, steps_c, torch.Generator().manual_seed(7)), rtol=1e-5, atol=1e-5)
    torch.manual_seed(2)
    eps_net_20 = wbd.Denoiser2D(50).eval()
    ab_20 = toy_schedule(np.linspace(1e-3, 0.2, 50))["alpha_bars"]
    for steps_c in (1, 4, 50):
        ok_20 &= same("B5.20", f"ddim_20 en {steps_c} pas contre le DDIM du contrat",
                      ddim_20(eps_net_20, 5, steps_c, ab_20, torch.Generator().manual_seed(8)),
                      wbd.sample_toy_ddim(eps_net_20, 5, steps_c, ab_20, torch.Generator().manual_seed(8)),
                      rtol=1e-5, atol=1e-5)
    fm_ok_20 = bool(ok_20)

In [ ]:
# The flow matching model, the same MLP and the same budget as in B5.19, then both samplers from 1 to 100 steps
MEASURED_20 = None
if not (fm_ok_20 and MODEL_19 is not None):
    print("⏳ Ex B5.20 : l'expérience attend tes trois fonctions (cellule précédente sans ❌) et le modèle de B5.19.")
else:
    torch.manual_seed(20)
    MODEL_FM_20 = Denoiser2D_19(wbd.FM_POSITIONS)
    start_20 = time.perf_counter()
    LOSSES_FM_20 = train_fm_20(MODEL_FM_20, DATA_B, SETTINGS_B["n_iters"], SETTINGS_B["batch_size"], SETTINGS_B["lr"],
                               torch.Generator().manual_seed(200))
    MODEL_FM_20.eval()
    print(f"entraînement du flow matching : {time.perf_counter() - start_20:.0f} s")
    MEASURED_20 = {"DDIM (bruit prédit, B5.19)": [], "Euler (vitesse prédite)": []}
    SAMPLES_20 = {}
    for n_20 in STEPS_20:
        pts_d = ddim_20(MODEL_19, 2000, n_20, SCHED_B["alpha_bars"], torch.Generator().manual_seed(201))
        pts_e = euler_20(MODEL_FM_20, 2000, n_20, torch.Generator().manual_seed(201))
        SAMPLES_20[n_20] = (pts_d, pts_e)
        MEASURED_20["DDIM (bruit prédit, B5.19)"].append(quality(pts_d)[0])
        MEASURED_20["Euler (vitesse prédite)"].append(quality(pts_e)[0])
    print("pas :", "  ".join(f"{n:>6}" for n in STEPS_20))
    for name, values in MEASURED_20.items():
        print(f"{name} :", "  ".join(f"{fr(v, 3):>6}" for v in values))
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))
    for (name, values), color in zip(MEASURED_20.items(), ("C0", "C1")):
        ax1.plot(STEPS_20, values, "o-", color=color, label=f"{name}, ton protocole")
        rows = np.array(FULL_20[name])                         # (seeds, steps) of the full protocol
        ax1.fill_between(STEPS_20, rows.min(axis=0), rows.max(axis=0), color=color, alpha=0.15)
    ax1.axhline(FLOOR_B, color="gray", lw=0.8, ls=":", label="deux échantillons de l'anneau")
    ax1.set_xscale("log")
    ax1.set_yscale("log")
    ax1.set_xlabel("nombre de pas d'échantillonnage")
    ax1.set_ylabel("distance de Wasserstein en tranches")
    ax1.legend(fontsize=7)
    for ax, (pts, title) in zip((ax2, ax3), ((SAMPLES_20[5][0], "DDIM, 5 pas"), (SAMPLES_20[5][1], "Euler, 5 pas"))):
        ax.scatter(*REAL_B.T, s=3, alpha=0.2, color="gray")
        ax.scatter(*np.clip(pts, -4, 4).T, s=3, alpha=0.4)
        ax.set_xlim(-4, 4)
        ax.set_ylim(-4, 4)
        ax.set_aspect("equal")
        ax.set_title(title)
    plt.show()
    print(f"protocole complet ({len(FULL_20['seeds'])} graines, bandes du graphique) : distance médiane par nombre de pas")
    for name in MEASURED_20:
        print(f"{name} :", "  ".join(f"{fr(v, 3):>6}" for v in np.median(np.array(FULL_20[name]), axis=0)))

In [ ]:
fewest_20 = ...     # a) the smallest number of steps of STEPS_20 from which Euler is within 3 times its distance at
                    #    100 steps, for every seed of the full protocol: an integer of STEPS_20
better5_20 = ...    # b) at 5 steps, which sampler has the smaller distance for every seed: "D" (DDIM) or "E" (Euler)?

In [ ]:
with wb.attempt("B5.20"):
    wb.check("B5.20a", fewest_20)
with wb.attempt("B5.20"):
    wb.check("B5.20b", better5_20)

## Partie C · Le U-Net sur MNIST et Fashion-MNIST

Place aux images. Tu écris d'abord le **U-Net minuscule** de la fiche (B5.21), que la suite de la partie entraîne sur MNIST, échantillonne, conditionne par la classe et évalue. Les images sont ramenées dans $[-1, 1]$ par une seule fonction, `wbd.to_signed` ($x / 127{,}5 - 1$ pour des octets) : le réseau, la forme fermée et l'écrêtage de $\hat{\mathbf{x}}_0$ supposent tous cette échelle, et `wbd.to_unit` les ramène dans $[0, 1]$ pour l'affichage. Les données d'entraînement sont un sous-ensemble **figé** (`wbd.image_subset`) : 10 000 images en mode rapide, les 60 000 en mode complet, toujours dans le même ordre, de sorte que les petits sous-ensembles sont inclus dans les grands.

### Ex B5.21 — Un U-Net minuscule en PyTorch : blocs convolutifs, connexions de saut, encodage du pas de temps 📦 ★★★ ⏱️ 50 min
**Objectif :** écrire le U-Net de la fiche, avec son encodage du pas et une classe facultative, et le valider étage par étage contre le modèle du contrat.  
**Prérequis :** Ex B5.19 · B3 (`positional_encoding`) · ch. 21 (convolutions de stride 2, convolution transposée) · fiche §205.4 (« Le U-Net et ses connexions de saut », « Un bloc résiduel conditionné par le pas ») · §205.7 (« Donner la classe au réseau ») · **Fil rouge :** MNIST · **Parcours :** R, C

**Contrat.** Les noms des modules comptent : la suite du chapitre sauvegarde ton modèle par son `state_dict` et le recharge.
- `ResBlock_21(c_in, c_out, t_dim)` : le bloc de la fiche (§205.4), avec ses six sous-modules `norm1` (`nn.GroupNorm(8, c_in)`), `conv1` (3×3, `padding=1`), `time` (`nn.Linear(t_dim, c_out)`), `norm2`, `conv2` et `skip` (une convolution 1×1 si `c_in != c_out`, sinon `nn.Identity()`) ; `forward(x, temb)` : $\mathbf{h}$ = `conv1(silu(norm1(x)))`, plus `time(temb)` en un biais par canal, puis `conv2(silu(norm2(h)))`, plus `skip(x)`.
- `TinyUNet_21(n_steps, channels=(32, 64, 128), t_dim=128, n_classes=None)`, avec `c1, c2, c3 = channels` :
  - `self.config`, le dictionnaire `{"n_steps", "channels" (une liste), "t_dim", "n_classes"}` de ses arguments : il suffit à reconstruire le modèle (`TinyUNet_21(**config)`) ;
  - un buffer **non persistant** `pe` : la table `positional_encoding(n_steps, t_dim)` de B3, en float32 ; `t_mlp = nn.Sequential(nn.Linear(t_dim, t_dim), nn.SiLU(), nn.Linear(t_dim, t_dim))` ;
  - si `n_classes` est un entier : `class_emb = nn.Embedding(n_classes + 1, t_dim)`, dont la **dernière** ligne, d'indice `n_classes`, est la classe vide $\varnothing$ de la guidance sans classifieur ;
  - `inp` (1 → `c1`, 3×3) ; l'encodeur `d1 = ResBlock_21(c1, c1)`, `down1` (convolution 3×3 de stride 2, `padding=1`, `c1` → `c1` : 28 → 14), `d2 = ResBlock_21(c1, c2)`, `down2` (`c2` → `c2`, 14 → 7) ; `mid = ResBlock_21(c2, c3)` ; le décodeur `up2` (`nn.ConvTranspose2d(c3, c2, 4, stride=2, padding=1)` : 7 → 14), `u2 = ResBlock_21(2 * c2, c2)`, `up1` (`c2` → `c1`, 14 → 28), `u1 = ResBlock_21(2 * c1, c1)` ; `out` (`c1` → 1, 3×3) ; tous les blocs reçoivent `t_dim` ;
  - `forward(x, t, y=None)` : `x` de forme `(N, 1, 28, 28)`, `t` des indices int64 `(N,)`, `y` des classes int64 `(N,)` de 0 à `n_classes` (la classe vide si `y` est `None`) ; `temb = t_mlp(pe[t])`, plus `class_emb(y)` si le modèle a des classes ; puis `h1 = d1(inp(x))`, `h2 = d2(down1(h1))`, `h = mid(down2(h2))`, `h = u2(cat([up2(h), h2]))`, `h = u1(cat([up1(h), h1]))` (la concaténation sur l'axe des canaux, **la sortie du décodeur d'abord**), et `out(h)` : la prédiction du bruit, `(N, 1, 28, 28)`.

Puis, dans la même cellule, avec ton modèle :
- a) `n_small_21` : le nombre de paramètres de `TinyUNet_21(400, channels=(16, 32, 64), t_dim=64)` ;
- b) `u1_in_21` : la forme de l'entrée de `u1` (après la concaténation) pour ce modèle et un batch de 8 images, un tuple de 4 entiers.

La cellule de vérification compare ton bloc à celui du contrat aux mêmes poids, puis, pour trois réglages (celui de la fiche, `(16, 32, 64)` avec `t_dim=64`, et 10 classes), les noms et les formes du `state_dict`, le buffer `pe`, et la sortie de ton U-Net à celle du U-Net du contrat (`wbd.TinyUNet`) aux mêmes poids, éloignés de l'initialisation ; elle compare enfin la forme de chaque étage (par des *hooks*, `wbd.stage_shapes`) et vérifie tes deux réponses. La cellule suivante choisit le modèle de la suite du chapitre, `UNET_CLASS` : le tien s'il a tout passé, sinon celui du contrat, et le dit ; elle mesure une passe sur un batch de 64 images (une mesure, pas une réponse).

Dans tes notes : combien de canaux reçoit `u2`, et pourquoi la convolution qui suit doit-elle les attendre (fiche, piège « deux *skip connections* ») ? Que perdrait le U-Net sans ses connexions de saut ? Pourquoi une group norm plutôt qu'une batchnorm ici ? Pourquoi `pe` est-il un buffer non persistant, et que se passerait-il en rechargeant un `state_dict` avec un autre `n_steps` ? D'après ta mesure, combien de temps coûteraient 400 pas de DDPM pour 64 images, et 50 pas de DDIM ?

In [ ]:
class ResBlock_21(nn.Module):
    """Residual block conditioned by the step (contract of the statement)."""

    def __init__(self, c_in, c_out, t_dim):
        super().__init__()
        raise NotImplementedError("ResBlock_21")

    def forward(self, x, temb):
        raise NotImplementedError("ResBlock_21.forward")


class TinyUNet_21(nn.Module):
    """A tiny U-Net that predicts the noise of x_t (contract of the statement)."""

    def __init__(self, n_steps, channels=(32, 64, 128), t_dim=128, n_classes=None):
        super().__init__()
        raise NotImplementedError("TinyUNet_21")

    def forward(self, x, t, y=None):
        raise NotImplementedError("TinyUNet_21.forward")


n_small_21 = ...    # a) the number of parameters of TinyUNet_21(400, channels=(16, 32, 64), t_dim=64), with your model
u1_in_21 = ...      # b) the shape of the input of u1 (after the concatenation) for that model and a batch of 8 images,
                    #    a tuple of 4 integers

In [ ]:
UNET_OK_21 = False
with checked("B5.21"):
    torch.manual_seed(0)
    ok_21 = True
    blk_21, ref_blk = ResBlock_21(16, 32, 24), wbd.ResBlock(16, 32, 24)
    with torch.no_grad():
        for p_ in ref_blk.parameters():
            p_.add_(0.2 * torch.randn(p_.shape))
    if {k: tuple(v.shape) for k, v in blk_21.state_dict().items()} == \
            {k: tuple(v.shape) for k, v in ref_blk.state_dict().items()}:
        blk_21.load_state_dict(ref_blk.state_dict())
        xb_21, tb_21 = torch.randn(2, 16, 6, 6), torch.randn(2, 24)
        with torch.no_grad():
            ok_21 &= same("B5.21", "ResBlock_21(16, 32, 24) contre le bloc du contrat, aux mêmes poids", blk_21(xb_21, tb_21),
                          ref_blk(xb_21, tb_21), rtol=1e-5, atol=1e-5)
    else:
        ok_21 = False
        print(f"❌ Ex B5.21 : ResBlock_21(16, 32, 24) doit avoir les paramètres "
              f"{sorted(ref_blk.state_dict())} (noms et formes du contrat) ; reçu {sorted(blk_21.state_dict())}.")
    for kw_21 in ({}, {"channels": (16, 32, 64), "t_dim": 64}, {"n_classes": 10}):
        mine_21, ref_21 = TinyUNet_21(T_STEPS, **kw_21), wbd.TinyUNet(T_STEPS, **kw_21)
        shapes_m = {k: tuple(v.shape) for k, v in mine_21.state_dict().items()}
        shapes_r = {k: tuple(v.shape) for k, v in ref_21.state_dict().items()}
        if shapes_m != shapes_r:
            ok_21 = False
            diff_21 = sorted(set(shapes_m.items()) ^ set(shapes_r.items()))[:4]
            print(f"❌ Ex B5.21 : TinyUNet_21({T_STEPS}, {kw_21}) : le state_dict n'a pas les noms et les formes du "
                  f"contrat (premières différences, attendu ou reçu : {diff_21}).")
            continue
        pe_21 = getattr(mine_21, "pe", None)
        ok_21 &= same("B5.21", f"le buffer pe ({kw_21 or 'réglage par défaut'}) : la table de positional_encoding",
                      pe_21 if pe_21 is not None else torch.zeros(0), torch.tensor(wbd.sinusoidal_table(
                          T_STEPS, kw_21.get("t_dim", 128)), dtype=torch.float32), rtol=1e-5, atol=1e-6)
        with torch.no_grad():
            for p_ in ref_21.parameters():
                p_.add_(0.05 * torch.randn(p_.shape))                # weights far from the initialisation
        mine_21.load_state_dict(ref_21.state_dict())
        x_21, t_21 = torch.randn(3, 1, 28, 28), torch.tensor([0, T_STEPS // 2, T_STEPS - 1])
        with torch.no_grad():
            if "n_classes" in kw_21:
                y_21 = torch.tensor([3, 10, 7])
                ok_21 &= same("B5.21", "U-Net conditionné (y donné, dont la classe vide 10) contre le contrat",
                              mine_21(x_21, t_21, y_21), ref_21(x_21, t_21, y_21), rtol=1e-4, atol=1e-5)
                ok_21 &= same("B5.21", "U-Net conditionné, y = None : la classe vide", mine_21(x_21, t_21),
                              ref_21(x_21, t_21, torch.full((3,), 10)), rtol=1e-4, atol=1e-5)
            else:
                ok_21 &= same("B5.21", f"U-Net {kw_21 or '(32, 64, 128)'} contre le contrat, aux mêmes poids",
                              mine_21(x_21, t_21), ref_21(x_21, t_21), rtol=1e-4, atol=1e-5)
        if not kw_21:
            got_s, want_s = wbd.stage_shapes(mine_21, x_21, t_21), wbd.stage_shapes(ref_21, x_21, t_21)
            ok_21 &= got_s == want_s
            verdict("B5.21", got_s == want_s, "les formes de chaque étage sont celles du contrat.",
                    f"formes des étages : attendu {want_s}, reçu {got_s}.")
    UNET_OK_21 = bool(ok_21)
with wb.attempt("B5.21"):
    wb.check("B5.21a", n_small_21)
with wb.attempt("B5.21"):
    wb.check("B5.21b", shape_text(u1_in_21))

In [ ]:
# The model of part C: your U-Net if it passed its checks, otherwise the U-Net of the contract (wb.diffusion)
UNET_CLASS = TinyUNet_21 if UNET_OK_21 else wbd.TinyUNet
if not UNET_OK_21:
    print("ℹ️ Ex B5.21 : ton U-Net n'est pas encore écrit, ou n'a pas passé ses contrôles ; la suite du chapitre prend "
          "celui du contrat (wbd.TinyUNet). Relance cette cellule une fois le tien prêt.")
net_21 = UNET_CLASS(T_STEPS).eval()
x_21, t_21 = torch.randn(64, 1, 28, 28), torch.randint(0, T_STEPS, (64,))
with torch.no_grad():
    net_21(x_21, t_21)
    times_21 = []
    for _ in range(3):
        start_21 = time.perf_counter()
        net_21(x_21, t_21)
        times_21.append(time.perf_counter() - start_21)
SECONDS_21 = min(times_21)
print(f"{'ton' if UNET_OK_21 else 'le'} U-Net : une passe sur un batch de 64 images en {fr(SECONDS_21, 3)} s (le "
      f"meilleur de trois) ; {T_STEPS} pas de DDPM pour 64 images coûteraient ≈ {T_STEPS * SECONDS_21:.0f} s.")

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu calculer à la main un planning de bruit, ses $\bar\alpha_t$ et son rapport signal sur bruit ?
2. Sais-tu dériver la forme fermée et le posterior d'un pas, puis calculer un pas DDPM et un pas DDIM ?
3. Sais-tu appliquer la guidance sans classifieur et traduire `guidance_scale` en $w$ ?
4. Sais-tu estimer le coût d'un échantillonnage, et ce que gagnent DDIM, la distillation et l'espace latent ?

**Pour aller plus loin** : l'article de Ho, Jain et Abbeel et le billet de L. Weng, cités dans la fiche. La suite : le bonus B6, l'explicabilité et l'équité.

*Les exercices B5.22 à B5.31 (bugs, entraînement du U-Net sur MNIST, échantillonnage, guidance, mémorisation, distance de Fréchet, script autonome et défi) seront ajoutés à la prochaine étape de la génération ; relance alors `python tools/start_chapter.py B5` pour obtenir le notebook complet.*